In [ ]:
# @title
from IPython.core.display import display, HTML
import time
start_time = time.time()

display(HTML('''
<div style="display: flex; justify-content: space-between; align-items: center; padding: 10px;">
  <div style="flex: 1;">
    <h1 style="margin: 0;">Data Mining Project - Colorectal cancer</h1>
    <h2 style="margin: 0;">Data Mining 2</h2>
    <p style="margin: 5px 0;">Spring  Semester 2024-2025</p>
    <p style="margin: 5px 0;">Group 33</p>
    <p style="margin: 5px 0;">Created by:</p>
    <ul style="margin: 5px 0; padding-left: 20px;">
      <li>Bruno Chaves - 20240370</li>
      <li>Patrick Perlinger - 20240378</li>
      <li>Rodrigo Santos - 20240389</li>
      <li>Emmanuel Tronche-Macaire - 20240385</li>
    </ul>
  </div>
  <div style="flex: 0 0 auto; text-align: right;">
    <img src="https://urbandatalab.pt/images/partners/nova_ims.png" alt="Art Image" width="200" style="border-radius: 5px;" />
  </div>
</div>
'''))


# Preliminary settings and imports

In [2]:
# Importing python packages
# Standard Library
import time
from math import ceil
from itertools import product, combinations
from collections import defaultdict
from datetime import datetime

# Google Colab
from google.colab import drive, files

# Scientific Computing and Data Handling
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import chi2_contingency, pointbiserialr

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import graphviz

# Statsmodels
import statsmodels.api as sm
from statsmodels.formula.api import ols

# SymPy (used for physics/units)
from sympy.physics.units import length

# Scikit-learn: Model Selection
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RandomizedSearchCV
)

# Scikit-learn: Preprocessing
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer

# Scikit-learn: Feature Selection
from sklearn.feature_selection import RFE

# Scikit-learn: Linear Models
from sklearn.linear_model import LogisticRegression, LinearRegression

# Scikit-learn: Tree-based Models
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
)

# Scikit-learn: Other Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

# Scikit-learn: Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    r2_score, mean_absolute_error, mean_squared_error,
    confusion_matrix, classification_report, balanced_accuracy_score,
    RocCurveDisplay, root_mean_squared_error, make_scorer
)

# Pandas utilities
from pandas import value_counts

# Suppress warnings
import warnings



In [5]:
path = "https://raw.githubusercontent.com/emmacaire/Classification-academic-python/main/source/patient_train_data.csv"

# Reading the csv file
data = pd.read_csv(path)

In [6]:
# Reading the csv file

data = pd.read_csv(path)
data_copy_du = data.copy()

# Data Understanding

##Basic Tabular Exploration

In [7]:
# Understanding the shape of the dataframe

data_copy_du.shape

(75035, 32)

In [ ]:
# Some information about the amount of data entries for each column and the data types

data_copy_du.info()

In [ ]:
# Showing the first 5 rows of the df and not hiding any column in order to get a first feeling for the data we are facing

pd.set_option('display.max_columns', None)
data_copy_du.head()

In [ ]:
# Same thing as above, just instead of the first 5 rows we check the last 5 rows

data_copy_du.tail()

In [ ]:
# Get some information about the characteristics of our metric features

data_copy_du.describe()

# Get a table transposed
data_copy_du.describe().T

In [ ]:
# We define a list with all the metric features that are showing up in the table above (note: Marital Status is not a metric feature). This will be relevant for further analysis.

metric_features = ["Healthcare Costs", "Incidence Rate per 100K", "Mortality Rate per 100K", "Tumor Size (mm)"]

In [ ]:
# Get some information about the characteristics of our non-metric features

data_copy_du.describe(include = ['O'])
data_copy_du.describe(include = ['O']).T

In [ ]:
# As we could see in the overview above, most of the columns have some missing values. Let's display this in a better way in order to understand which columns have the most missing values.

data_copy_du.isna().sum()

In [ ]:
# We also want to understand if we have duplicated rows in our dataset
# For that, we check duplicates based on all the features except the "ID" column

data_copy_du.loc[:, data_copy_du.columns != 'ID'].duplicated().sum()

# As we can see, we have 15 duplicated rows in our dataset

##Univariate Visual Exploration

In [ ]:
# Drop ID column
data_copy_du.drop('ID', axis=1, inplace=True)

## Now let's focus on the visual representations of the distribution of our variables, in order to get a better understanding for them. Therefore we need to create some functions.

# Define a function that plots multiple histograms

def plot_multiple_histograms(data, feats, title="Numeric Variables' Histograms", figsize=(18, 8)):

    # Prepare figure. Create individual axes where each histogram will be placed
    fig, axes = plt.subplots(2, ceil(len(feats) / 2), figsize=figsize)

    # Plot data
    # Iterate across axes objects and associate each histogram (hint: use the ax.hist() instead of plt.hist()):
    for ax, feat in zip(axes.flatten(), feats): # Notice the zip() function and flatten() method
      ax.hist(data[feat])
      ax.set_title(feat)

    # Layout
    # Add a centered title to the figure:
    plt.suptitle(title)

    plt.show()

    return


# Define a function that plots multiple box plots

def plot_multiple_boxplots(data, feats, title="Numeric Variables' Box Plots", figsize=(18, 8)):

    # Prepare figure. Create individual axes where each histogram will be placed
    fig, axes = plt.subplots(2, ceil(len(feats) / 2), figsize=figsize)

    # Plot data
    # Iterate across axes objects and associate each histogram (hint: use the ax.hist() instead of plt.hist()):
    for ax, feat in zip(axes.flatten(), feats): # Notice the zip() function and flatten() method
      sns.boxplot(x=data[feat], ax=ax)
      ax.set_title(feat)
      ax.set_xlabel("")


    # Layout
    # Add a centered title to the figure:
    plt.suptitle(title)

    plt.show()

    return

In [ ]:
# Let's plot the histograms for our metric features in order to visualize their distribution

column_names = data_copy_du.columns.tolist()
# All Numeric Variables' Histograms in one figure

# set the default styles
sns.set()

plot_multiple_histograms(data_copy_du, metric_features)

In [ ]:
# The histograms already generate an understanding of the range where most of our data points are situated and that there are some outliers present in our dataset.
# In order to visualize the outliers better, let's plot some boxplots

plot_multiple_boxplots(data_copy_du, metric_features)

In [ ]:
# So far we only visualized only 4 of our variables, as those are the only metric features we have in our dataset. Let's look at the non-metric features to gain better understanding for those too.

non_metric_features_df = data_copy_du.drop(columns=metric_features).drop(columns=["Date of Birth"])
# ID needs to be dropped as it is irrelevant and Date of Birth needs to be dropped as it would show too many categories in the countplot
non_metric_features_df.head()

non_metric_features = non_metric_features_df.columns.tolist()
print(non_metric_features)

In [ ]:
# Defining a function for a countplot in order to visualize all our non-metric (= categorical) features

def visualize_countplots(df, features):
    """
    Visualizes countplots for multiple features in a DataFrame.

    Parameters:
    df (pd.DataFrame): The DataFrame containing the data.
    features (list): A list of column names to visualize.

    Returns:
    None
    """
    num_features = len(features)
    fig, axes = plt.subplots(nrows=num_features, ncols=1, figsize=(10, 5 * num_features))

    for i, feature in enumerate(features):
        sns.countplot(data=df, x=feature, ax=axes[i])
        axes[i].set_title(f'Countplot of {feature}')
        axes[i].set_xlabel(feature)
        axes[i].set_ylabel('Count')

    plt.tight_layout()
    plt.show()

# Calling the function:
visualize_countplots(data_copy_du, non_metric_features)

These countplots show us very well how the data is distributed between the different categories of each variable and also gives us many insights about what will need to be done to preprocess the data.

##Exploration of Correlations

In [ ]:
# Let's use the Pearson Correlation Matrix to get an understanding of the relations between our metric features

def plot_corr_pearson(df, feats, title="Pearson Correlation Matrix", method="pearson", figsize=(10,8)):
  fig = plt.figure(figsize=figsize)
  corr = np.round(df[feats].corr(method=method), decimals=3)
  mask_annot = True
  annot = np.where(mask_annot, corr.values, np.full(corr.shape,""))
  sns.heatmap(data=corr, annot=annot, cmap=sns.diverging_palette(220, 10, as_cmap=True),
              fmt='s', vmin=-1, vmax=1, center=0, square=True, linewidths=.5)
  fig.subplots_adjust(top=0.95)
  fig.suptitle(title, fontsize=20)
  plt.show()
  return

plot_corr_pearson(data_copy_du, metric_features)

In [ ]:
# We can see that using the Pearson Correlation it seems like none of the metric features are in any way correlated to each other.
# Let's use the Spearman Correlation to understand if it shows similar results or not

def plot_corr_spearman(df, feats, title="Spearman Correlation Matrix", method="spearman", figsize=(10,8)):
  fig = plt.figure(figsize=figsize)
  corr = np.round(df[feats].corr(method=method), decimals=3)
  mask_annot = True
  annot = np.where(mask_annot, corr.values, np.full(corr.shape,""))
  sns.heatmap(data=corr, annot=annot, cmap=sns.diverging_palette(220, 10, as_cmap=True),
              fmt='s', vmin=-1, vmax=1, center=0, square=True, linewidths=.5)
  fig.subplots_adjust(top=0.95)
  fig.suptitle(title, fontsize=20)
  plt.show()
  return

plot_corr_spearman(data_copy_du, metric_features)

As the Spearman Correlation Matrix shows the same results as the Pearson Correlation Matrix, we can conclude that there are no relevant correlations between any of our metric features.

In [ ]:
# Now we want to understand the correlations between our non-metric features using the chi squared test
# First, let's have a look again at our non-metric features

non_metric_features

In [ ]:
# As we have seen previously, "Marital Status" has no values and "Transfusion History" only has one unique value. Therefore we exclude them from the chi squared test.

items_to_remove = {"Marital Status", "Transfusion History"}
non_metric_features_chisquared = [item for item in non_metric_features if item not in items_to_remove]
print(non_metric_features_chisquared)


In [ ]:
# Defining the function that will create a list of the results of the chi squared test

def chi_squared_test(df, variables):
    results = []

    for var1, var2 in combinations(variables, 2):
        contingency_table = pd.crosstab(df[var1], df[var2])

        # Skip if the table is empty
        if contingency_table.size == 0:
            continue

        chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

        # Append results as a dictionary
        results.append({
            "Variable 1": var1,
            "Variable 2": var2,
            "Chi-Square": chi2,
            "p-value": p,
            "Degrees of Freedom": dof,
            "Significant": p < 0.05  # Boolean indicating significance
        })

    # Convert list of dictionaries to DataFrame
    return pd.DataFrame(results)

# Call the function
chi_squared_results = chi_squared_test(data_copy_du, non_metric_features_chisquared)
print(chi_squared_results)

In [ ]:
# We can see that due to the amount of non-metric features we got 276 rows. But we are only interested in the ones where "Significant" == True. Let's look at these.

significant_results = chi_squared_results[chi_squared_results["Significant"] == True]
print(significant_results)

In contrast to the metric features, we found some correlations between the non-metric features. In total we have 9 significant correlations, one of them being a correlation to the target variable "Survival Prediction"

In [ ]:
# Ultimately, we also want to understand the correlations between metric and non-metric variables.
# For this we use the One Way ANOVA Test

# Defining a function that calculates the One Way ANOVA and shows the significant correlations in violin plots + prints some statistical data about each significant correlation
def anova_explorer(df, numeric_vars, categorical_vars, alpha=0.05):
    """
    Performs one way ANOVA between each pair of numeric and categorical variables.
    Prints statistics and shows violin plots for significant results (p-value < 0.05).
    """
    for num_var in numeric_vars:
        for cat_var in categorical_vars:
            # Drop rows with missing data
            sub_df = df[[num_var, cat_var]].dropna()

            # Group numeric variable by each category
            groups = [sub_df[num_var][sub_df[cat_var] == group] for group in sub_df[cat_var].unique()]

            if len(groups) < 2:
                continue  # Skip if not enough groups to compare

            # Perform one-way ANOVA
            f_stat, p_val = stats.f_oneway(*groups)

            if p_val < alpha:
                print(f"\nSignificant result for '{num_var}' vs '{cat_var}' (p = {p_val:.4f})")

                # Descriptive stats
                print(f"{'Category':<20} {'Mean':>10} {'Variance':>15} {'Sample Size':>15}")
                for group_name, group_data in sub_df.groupby(cat_var):
                    mean = group_data[num_var].mean()
                    var = group_data[num_var].var()
                    n = group_data[num_var].count()
                    print(f"{str(group_name):<20} {mean:10.2f} {var:15.2f} {n:15}")

                # Violin plot
                plt.figure(figsize=(8, 5))
                sns.violinplot(x=cat_var, y=num_var, data=sub_df)
                plt.title(f"Violin Plot: {num_var} by {cat_var}")
                plt.tight_layout()
                plt.show()

In [ ]:
#anova_explorer(data_copy_du, metric_features, non_metric_features_chisquared)

Performing the One Way ANOVA Test shows us that there are 7 significant correlations (p <0.05) between numeric and categorical variables. When looking at the Violin Plots and the Mean Values, we can see that none of these significant relationships results due to high differences in the means of each category, but rather due to the high sample sizes, that cause the p-value to be lower than 0.05, even if the percentage of the difference between the means is very small.
Therefore, while we can conclude that some significant relationships exist, the actual differences caused by these relationships are rather small.

## Data Preprocessing

In [ ]:
preprocessing_df = data.copy()

#Remove ID Column
preprocessing_df = preprocessing_df.loc[:, preprocessing_df.columns != 'ID']


## Duplicates removal

We notice that there are 15 rows that show the exact same values by pair. While this could happen for categorical value, it seems very unlikely that the numerical values also match exactly. Therefore we treat these rows as duplicates and remove one of the pair for each one. As a result, we remove a total of 15 rows from the dataset.


In [ ]:
print(f"Initial size of our data before duplicate removal: {preprocessing_df.shape}")

#print(preprocessing_df[preprocessing_df.duplicated()])
#Drop duplicates (keep the first occurrence, for example)
preprocessing_df = preprocessing_df.drop_duplicates(keep='first')

print(f"Final size of our preprocessing_df after duplicate removal: {preprocessing_df.shape}")

## Dataset split

The method we will use to split the dataset is the hold-out method, dividing the dataset of labeled data into 3 separate sets: 70% of values will form the training set, 15% the validation set and 15% the test set. According to our evaluation and data understanding, the size and type of data does not require k-fold cross-validation.

In [ ]:
data_var = preprocessing_df.loc[:, preprocessing_df.columns != 'Survival Prediction']
target_var = preprocessing_df['Survival Prediction']

In [ ]:
X_train, X_temp, y_train, y_temp  = train_test_split(data_var,
                                                            target_var,
                                                            test_size = 0.3,
                                                            random_state = 33,
                                                            shuffle = True,
                                                            stratify = target_var)

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(X_temp,
                                                y_temp,
                                                test_size=0.50,
                                                random_state=33,
                                                shuffle=True,
                                                stratify=y_temp)

In [ ]:
shape_check = np.array([
    ['data', data.shape],
    ['X_train', X_train.shape],
    ['X_val', X_val.shape],
    ['X_test', X_test.shape]
], dtype=object)

print(shape_check)

We now have three subsets of the original "data" dataset,
* X_train : our training set (70% of observations, 52523 total rows)
* X_val : our validation set (15% of observations, 11256 total rows)
* X_test : our test set (15% of observations, 11256 total rows)

We also have three corresponding subsets with the target variable only as column, and the same rows split as the sets above:
* y_train
* y_val
* y_test

We also make a copy of our original dataset before we start preprocessing the data.

In [ ]:
X_train_copy = X_train.copy()
X_val_copy = X_val.copy()
X_test_copy = X_test.copy()

y_train_copy = y_train.copy()
y_val_copy = y_val.copy()
y_test_copy = y_test.copy()



## Outliers

As shown in the boxplots of the univariate analysis in the data understanding section, the quantitative variables reveal a low number of outliers and all suggest a clear cut point between for all four variables, two of which only on the upper whisker and two on both sides of the plot. While negative values are clearly a wrong data entry, for other values we might suggest other hypothesis such as the use of a different unit of measure or an extra zero at the end. But given the low number of rows affected, we can safely remove those without impacting the quality of data in a significant way.

### Outlier Detection

In [ ]:
cols_to_remove_outliers = ['Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K', 'Tumor Size (mm)']

datasets = [X_train_copy]
dataset_names = ['Training Set']

n_total = len(X_train_copy)
print("Outliers removed per variable (count and % of total):")

#outlier detection
for dataset, dataset_name in zip(datasets, dataset_names):
    print(f"\nProcessing {dataset_name}:")
    for i in cols_to_remove_outliers:
        Q1 = X_train_copy[i].quantile(0.25)
        Q3 = X_train_copy[i].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # Identify outliers
        outliers = dataset[(dataset[i] < lower_bound) | (dataset[i] > upper_bound)]
        # Count of outliers
        n_outliers = len(outliers)
        print(f"Outliers from {i}: {n_outliers} ({(n_outliers / n_total * 100):.2f}%)")

### Outlier Removal

In [ ]:
print(X_train_copy.shape)
# Outlier removal
for col in cols_to_remove_outliers:
        Q1 = X_train_copy[col].quantile(0.25)
        Q3 = X_train_copy[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        X_train_copy = X_train_copy[(X_train_copy[col] >= lower_bound) & (X_train_copy[col] <= upper_bound)]

print(X_train_copy.shape)

#Apply changes made in X_train_copy to Y_test_copy
y_train_copy = y_train_copy[X_train_copy.index]

In [ ]:
# Visualize the updated datasets
#plot_multiple_boxplots(X_train_copy, cols_to_remove_outliers)
#plot_multiple_boxplots(X_val_copy, metric_features)
#plot_multiple_boxplots(X_test_copy, metric_features)


## Data inconsistencies

In this section we check the different categories that each variable has. Below are only reported the ones that have been modified and improved.
Here is a summary of the categories affected and transformed:
- Gender: re-assign value 'P' to the mode 'M'
- Healthcare Access: label '?' as missing value
- Smoker: we have two columns including smoking habits. Use the column without missing data and make the categories more intuitive by inverting 'Yes' and 'No'.
- Transfusion History: label '-' as missing value
- Urban or Rural: re-assign 'urban' to 'Urban' and 'rural' to 'Rural'

In [ ]:
# Group datasets for looping
datasets = [X_train_copy, X_val_copy, X_test_copy]

#**Gender**: we have three categories M, F, P corresponding to "male", "female" and "prefer not to say". The third one is considerably smaller than the others, therefore it makes sense to re-assign the values to one of the two categories. Considering the dominance of M, we will impute all 8 values "P" to the mode "M". This way there is a good probability that over 50% of the values labelled as "P" are correct in terms of gender assigned at birth. We reassign them in this section based on common sense, because it would be incorrect to treat them as missing values.

# --- Gender: replace 'P' with 'M' ---
for df in datasets:
    df['Gender'] = df['Gender'].replace('P', 'M')

#**Healthcare Access**: in this variable we have a few values registering '?' as an answer, we will for now impute them as missing values and analyze them in the following section.

# --- Healthcare Access: replace '?' with np.nan ---
for df in datasets:
    df['Healthcare Access'] = df['Healthcare Access'].replace('?', np.nan)

#**Smoker**: we have two duplicate columns, "Non Smoker" and "Smoking history". While the latter makes more common sense because "Yes" means smoker and "No" means non smoker, the former has all values, so using this one we do not have any missing data to address later. As it is, smokeres are labeled as "No" and non smokers as "Yes", which can be confusing. To make it more easy to read, we will rename the variable "Smoker" and change the responses accordingly. We assign 'Yes' values to a temporary 'temp' category otherwise, if we assigned them to 'No' immediately, when we move 'No' to 'Yes' they would be reassigned as well.

#We also need to remove the other variable 'Smoking History' which is redundant with 'Smoker' and has missing values.
# --- Smoker column: rename, relabel, and drop redundant column ---
for df in datasets:
    df.rename(columns={'Non Smoker': 'Smoker'}, inplace=True)
    # Relabel 'Yes' to 'No', 'No' to 'Yes'
    df['Smoker'] = df['Smoker'].replace({'Yes': 'temp', 'No': 'Yes'})
    df['Smoker'] = df['Smoker'].replace({'temp': 'No'})

#**Transfusion History**: all rows have '-' as entry. We need to label them as missing values and treat them in the next section.
# --- Transfusion History: replace '-' with np.nan ---
for df in datasets:
    df['Transfusion History'] = df['Transfusion History'].replace('-', np.nan)

#**Urban or Rural**: some categories are in capital letter, others not. We will assing the lower case to the capital ones.
# --- Urban or Rural: Normalize case and labels ---
for df in datasets:
    df['Urban or Rural'] = df['Urban or Rural'].str.lower()
    df['Urban or Rural'] = df['Urban or Rural'].replace({'urban': 'Urban', 'rural': 'Rural'})

## Missing values

Most of the variables have missing values.
Here is a summary of the categories affected and the methods to handle them:
- **column removal**: 2 variables have complete lack of data and are removed. (*'Marital Status', 'Transfusion History'*)
- **imputing to median**: all 4 quantitative variables have missing data for < 0.1% of the datasets. Therefore we can safely impute the median without altering too much the distribution. (*'Healthcare Costs', 'Incidence Rate per 100K', 'Mortality rate per 100K, 'Tumor Size (mm)'*)
- **imputing to mode**: for categorical variables that have a clear mode (majority class +40% of values than any other), imputing the mode is likely to be more accurate than imputing at random. (*'Country', 'Diabetes', 'Diet Risk', 'Early Detection', 'Family History', 'Gender', 'Genetic Mutation', 'Healthcare Access', 'Inflammatory Bowel Disease', 'Insurance Status', 'Urban or Rural'*)
- **imputing at random**: for categorical variables without a clear mode (no category has +40% more observations than any other variable), even though missing values account for less than 0.1% of the rows in the datasets, imputing to the mode can be distorting especially for those variables with many different categories, such as Country and Date of Birth. For this reason the values will be imputed at random. (*'Alcohol Consumption', 'Cancer Stage', 'Date of Birth', 'Obesity BMI', 'Physical Activity', 'Screening History', 'Treatment Type'*)



---



In [ ]:
datasets_names = [X_train_copy ,X_val_copy ,X_test_copy]
for i in datasets_names :
	print(f"\n\n {i.isna().sum()}")

In [ ]:
datasets_names = [X_train_copy, X_val_copy, X_test_copy]
for i in datasets_names :
	print(f"\n\n {(i.isna().sum() / len(i) * 100).round(2)}")

### Column Removal

- **Marital Status**: all values are missing. We drop the column.
- **Transfusion History**: all values are missing. We drop the column.
- **Diabetes History**: the data in this column is too imbalanced. We do not want to encode the variable with such imbalance. Additionally, re-assigning 'Yes' to the mode does not make sense either, as there would only be one category. For this reason we decide to drop the column.



In [ ]:
print(X_train_copy['Marital Status'].value_counts())
print(X_train_copy['Transfusion History'].value_counts())

datasets_names = [X_train_copy, X_val_copy, X_test_copy]
for i in datasets_names:
    i.drop(columns=['Marital Status'], inplace=True)
    i.drop(columns=['Transfusion History'], inplace=True)
    i.drop(columns=['Diabetes History'], inplace=True)
    i.drop(columns=['Smoking History'], inplace=True)

#X_val['Marital Status'].value_counts()
#X_test['Marital Status'].value_counts()
#X_val['Transfusion History'].value_counts()
#X_test['Transfusion History'].value_counts()

In [ ]:
print(X_train_copy['Healthcare Costs'].head())
X_train_copy.info()



---



### Imputation

#### Imputing to the median (quantitative variables)

We impute the median of the train dataset to all three datasets that we splitted. We choose median since it's less sensitive to outliers.

In [ ]:
#check data_fill_mean = data.fillna(data.mean(numeric_only=True))
#check data_fill_mean.isna().sum()

cols_to_impute_numeric = ['Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K', 'Tumor Size (mm)']

# List the columns you want to fill with their median

for col in cols_to_impute_numeric:
    X_train_copy.loc[:, col] = X_train_copy[col].fillna(X_train_copy[col].median())
    X_test_copy.loc[:, col] = X_test_copy[col].fillna(X_train_copy[col].median())
    X_val_copy.loc[:, col] = X_val_copy[col].fillna(X_train_copy[col].median())

#### Imputing to the mode (qualitative variables)
> With a clear mode

In [ ]:
columns = ['Country', 'Diabetes', 'Diet Risk', 'Early Detection', 'Family History', 'Gender', 'Genetic Mutation', 'Healthcare Access','Inflammatory Bowel Disease', 'Insurance Status', 'Urban or Rural']

#for col in columns:
    #print(f"\n=== {col} ===")
    #print("Train:\n", X_train[col].value_counts())
    #print("Val:\n", X_val[col].value_counts())
    #print("Test:\n", X_test[col].value_counts())

In [ ]:
impute_with_mode_var = ['Country', 'Diabetes', 'Diet Risk', 'Early Detection', 'Family History', 'Gender', 'Genetic Mutation','Healthcare Access', 'Inflammatory Bowel Disease', 'Insurance Status', 'Urban or Rural']

for col in impute_with_mode_var:
    X_train_copy[col] = X_train_copy[col].fillna(X_train_copy[col].mode()[0])
    X_val_copy[col] = X_val_copy[col].fillna(X_train_copy[col].mode()[0])
    X_test_copy[col] = X_test_copy[col].fillna(X_train_copy[col].mode()[0])



---



#### Imputing at random (qualitative variables)
> without a clear mode

In [ ]:
columns = ['Alcohol Consumption','Cancer Stage','Date of Birth','Obesity BMI','Physical Activity','Screening History','Treatment Type']

for col in columns:
    print(f"\n=== {col} ===")
    print("Train:\n", X_train_copy[col].value_counts())
    print("Val:\n", X_val_copy[col].value_counts())
    print("Test:\n", X_test_copy[col].value_counts())

In [ ]:
impute_with_random_var = ['Alcohol Consumption', 'Cancer Stage', 'Date of Birth', 'Obesity BMI', 'Physical Activity', 'Screening History', 'Treatment Type']

for col in impute_with_random_var:
    non_missing = X_train_copy[col].dropna().values
    X_train_copy[col] = X_train_copy[col].apply(lambda x: np.random.choice(non_missing) if pd.isna(x) else x)
    X_val_copy[col] = X_val_copy[col].apply(lambda x: np.random.choice(non_missing) if pd.isna(x) else x)
    X_test_copy[col] = X_test_copy[col].apply(lambda x: np.random.choice(non_missing) if pd.isna(x) else x)

In [ ]:
# Final check that all the missing values have been filled
#X_train_copy.isna().sum()
#X_val_copy.isna().sum()
#X_test_copy.isna().sum()

## Encoding

At this stage we need to transform all categorical variables into some values that can be later processed during the model selection and training. We will use the following strategies:
- column removal for a very imbalanced binary variable
- one-hot encoding for binary variables (including the target variable) and non binary variables that are not ordinal
- encoding from 0 to k in increasing order, for non binary variables
- average target encoding for a variable that has 16 different categories

### Encoding of the target variable

First of all we need to encode the categories in the dataset with the target variable, changing 'Yes' to 1 and 'No' to 0.

In [ ]:
# List of target Series
series_target = [y_train_copy, y_val_copy, y_test_copy]

# Replace 'Yes' with 1 and 'No' with 0 for each target
for i in series_target:
    i.replace({'Yes': 1, 'No': 0}, inplace=True)

### Binary encoding (Yes/No categories)

As a rule, we encode 'Yes' answers with 1 and 'No' with 0. List of variables involved:
- Alcohol Consumption
- Diabetes
- Early Detection
- Family History
- Genetic Mutation
- Heart Disease History
- Hypertension
- Inflammatory Bowel Disease
- Smoker

In [ ]:
yes_no_columns = [
    'Alcohol Consumption',
    'Diabetes',
    'Early Detection',
    'Family History',
    'Genetic Mutation',
    'Heart Disease History',
    'Hypertension',
    'Inflammatory Bowel Disease',
    'Smoker']

# Loop over each column and apply the mapping in all datasets
for col in yes_no_columns:
    for df in [X_train_copy, X_val_copy, X_test_copy]:
        df[col] = df[col].replace({'No': 0, 'Yes': 1})

### One-hot encoding (binary variables, other than Yes/No)

For other binary variables where we do not have Yes/No answers, it might be more complicated to replicate the association Yes = 1 and No = 0, since the attribution of 0 and 1 might be more arbitrary and less intuitive. For this reason we apply one-hot encoding here too, and this will generate one column that will replace the original one. List of variables involved:
- Gender
- Insurance Status
- Urban or Rural


In [ ]:
X_train_copy = pd.get_dummies(X_train_copy, columns=['Gender'], prefix='Gender', drop_first=True)
X_val_copy = pd.get_dummies(X_val_copy, columns=['Gender'], prefix='Gender', drop_first=True)
X_test_copy = pd.get_dummies(X_test_copy, columns=['Gender'], prefix='Gender', drop_first=True)

X_train_copy = pd.get_dummies(X_train_copy, columns=['Insurance Status'], prefix='Insurance Status', drop_first=True)
X_val_copy = pd.get_dummies(X_val_copy, columns=['Insurance Status'], prefix='Insurance Status', drop_first=True)
X_test_copy = pd.get_dummies(X_test_copy, columns=['Insurance Status'], prefix='Insurance Status', drop_first=True)

X_train_copy = pd.get_dummies(X_train_copy, columns=['Urban or Rural'], prefix='Urban or Rural', drop_first=True)
X_val_copy = pd.get_dummies(X_val_copy, columns=['Urban or Rural'], prefix='Urban or Rural', drop_first=True)
X_test_copy = pd.get_dummies(X_test_copy, columns=['Urban or Rural'], prefix='Urban or Rural', drop_first=True)

### One-hot encoding (non binary variables, non ordinal)

For variables that have more than two categories (non binary) and are of non ordinal nature, we need to create k-1 columns (for k categories) to be able to process the data.
List of variables involved:
- Treatment Type

In [ ]:
X_train_copy = pd.get_dummies(X_train_copy, columns=['Treatment Type'], prefix='Treatment Type', drop_first=True)
X_val_copy= pd.get_dummies(X_val_copy, columns=['Treatment Type'], prefix='Treatment Type', drop_first=True)
X_test_copy = pd.get_dummies(X_test_copy, columns=['Treatment Type'], prefix='Treatment Type', drop_first=True)

The code has created three new columns, 'Treatment Type_Combination', 'Treatment Type_Radiotherapy' and 'Treatment Type_Surgery'. When all are False it means the category is 'Chemotherapy'.

In [ ]:
X_train_copy.head()

### Ordinal encoding (non binary variables, ordinal)

For variables that have more than two categories (non binary) and but can be reasonably ordered based on common sense, we do not need one-hot encoding with the addition of new columns but we can simply order the values in increasing order from 0 onwards.
List of variables involved:
- Cancer Stage
- Diet Risk
- Healthcare Access
- Insurance Costs
- Obesity BMI
- Physical Activity
- Screening History

In [ ]:
# Define the maps
mappings = {
    'Diet Risk': {'Low': 0, 'Moderate': 1, 'High': 2},
    'Cancer Stage': {'Localized': 0, 'Regional': 1, 'Metastatic': 2},
    'Healthcare Access': {'Low': 0, 'Moderate': 1, 'High': 2},
    'Insurance Costs': {'No insurance': 0, 'Basic': 1, 'Extended': 2},
    'Obesity BMI': {'Normal': 0, 'Overweight': 1, 'Obese': 2},
    'Physical Activity': {'Low': 0, 'Moderate': 1, 'High': 2},
    'Screening History': {'Never': 0, 'Irregular': 1, 'Regular': 2}}

# List of datasets
datasets = [X_train_copy, X_val_copy, X_test_copy]

# Apply the mappings using a loop
for col, mapping in mappings.items():
    for df in datasets:
        df[col] = df[col].replace(mapping)

### Average Target Encoding

For variable 'Country', which shows 16 different categories, we opt for average target encoding to avoid creating 15 additional columns with one-hot encoding.

In [ ]:
X_train['Country'].value_counts()

In [ ]:
# Combine features and target into a single DataFrame
country_X_train = X_train_copy.copy()
country_X_train['Target'] = y_train_copy.copy()

country_X_val = X_val_copy.copy()
country_X_val['Target'] = y_val_copy.copy()

country_X_test = X_test_copy.copy()
country_X_test['Target'] = y_test_copy.copy()

# Compute mean target per country
country_target_mean_X_train = country_X_train.groupby('Country')['Target'].mean()
country_target_mean_X_val = country_X_val.groupby('Country')['Target'].mean()
country_target_mean_X_test = country_X_test.groupby('Country')['Target'].mean()

# Replace 'Country' values with their corresponding target mean
X_train_copy['Country enc'] = X_train_copy['Country'].map(country_target_mean_X_train)
X_val_copy['Country enc'] = X_val_copy['Country'].map(country_target_mean_X_val)
X_test_copy['Country enc'] = X_test_copy['Country'].map(country_target_mean_X_test)

#Optional (would not do at this stage...)
#Dropping the previous 'Country' column
#X_train_copy = X_train_copy.drop(columns=['Country'])
#X_val_copy = X_val_copy.drop(columns=['Country'])
#X_test_copy = X_test_copy.drop(columns=['Country'])

In [ ]:
#X_train_copy['Country enc'].value_counts()
#X_val_copy['Country enc'].value_counts()
#X_test_copy['Country enc'].value_counts()

## Feature Engineering

Creating 'Age' from 'Date of Birth'

We transform the data in the column 'Date of Birth' into age, which is easier to read.

In [ ]:
# Convert 'Date of Birth' to datetime (format: dd-mm-yyyy)
X_train_copy['Date of Birth'] = pd.to_datetime(X_train_copy['Date of Birth'], format='%d-%m-%Y', errors='coerce')
X_val_copy['Date of Birth'] = pd.to_datetime(X_val_copy['Date of Birth'], format='%d-%m-%Y', errors='coerce')
X_test_copy['Date of Birth'] = pd.to_datetime(X_test_copy['Date of Birth'], format='%d-%m-%Y', errors='coerce')

# Get today's date
today = pd.to_datetime('today')

# Compute age accurately (without rounding up)
X_train_copy['Age'] = X_train_copy['Date of Birth'].apply(lambda dob:
    today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day)) if pd.notnull(dob) else None)
X_val_copy['Age'] = X_val_copy['Date of Birth'].apply(lambda dob:
    today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day)) if pd.notnull(dob) else None)
X_test_copy['Age'] = X_test_copy['Date of Birth'].apply(lambda dob:
    today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day)) if pd.notnull(dob) else None)

# Convert to integer type (optional, drop rows with NaT if needed)
X_train_copy['Age'] = X_train_copy['Age'].astype('Int64')
X_val_copy['Age'] = X_val_copy['Age'].astype('Int64')
X_test_copy['Age'] = X_test_copy['Age'].astype('Int64')

# Drop 'Date of Birth' if you no longer need it
# data = data.drop(columns=['Date of Birth'])

Creating new categorical columns that might be of use to replace some existing ones

In [ ]:
## ------------ NEW Categorical  Columns ----------------------
datasets_names = [X_train_copy, X_val_copy, X_test_copy]

### New column with Age Categories named "age_group"
for i in datasets_names :
    i["Age_groups"] = pd.cut(x=i['Age'], bins=[0,30,60,100], labels=[0,1,2]) #Bins related to age 0 = "Young", 1 = "Middle_aged", 2 = "Old"

In [ ]:
### New column 'Tumor Size Categories': Discretize 'Tumor Size (mm)' into categories (small, medium, large).
##https://pubmed.ncbi.nlm.nih.gov/20101166/   --------->  Reference for tumor classification
for i in datasets_names :
    i["Tumor Size Categories"] = pd.cut(x=i['Tumor Size (mm)'], bins=[0,35,45,100], labels=[0,1,2])  #Bins related to tumor size in mm 0 = "Small", 1 = "Medium", 2 = "Large"


In [ ]:
### New column with 'Smoker and Alcohol Consumption' Boolean variable if a patient is both a smoker and alcohol consumer
for i in datasets_names :
    i["Smoker and Alcohol Consumption"] = ((i['Smoker'] == 1) & (i['Alcohol Consumption'] == 1))

In [ ]:
### New column with 'Comorbidity Count' (Sum binary columns representing existing conditions like 'Diabetes', 'Heart Disease History', 'Hypertension', 'Inflammatory Bowel Disease'), Comorbidity is the simultaneous presence of two or more diseases or medical conditions in a patient
comorbidity_cols = ['Diabetes', 'Heart Disease History', 'Hypertension', 'Inflammatory Bowel Disease']

for i in datasets_names:
    i['Comorbidity Count'] = i[comorbidity_cols].sum(axis=1)

In [ ]:
## ------------ NEW Numerical Columns ----------------------
### New column 'Mortality_to_Incidence_Ratio_per_100k': Divide 'Mortality Rate per 100K' by 'Incidence Rate per 100K'.
for i in datasets_names:
    i['Mortality_to_Incidence_Ratio_per_100k'] = round(i['Mortality Rate per 100K'] / i['Incidence Rate per 100K'],2)

In [ ]:
### New column 'Family History and Genetic Mutation Interaction': Boolean if both 'Family History' and 'Genetic Mutation' are present.
for i in datasets_names :
    i["Family History and Genetic Mutation Interaction"] = ((i['Family History'] == 1) & (i['Genetic Mutation'] == 1))

## Correlations & Associations

To avoid data leakage, calculations are performed only on the **training set**

In [ ]:
data_correlations = X_train_copy.copy()
data_correlations['Survival Prediction'] = y_train_copy

boolean_columns = ['Survival Prediction', 'Alcohol Consumption','Diabetes','Early Detection','Family History','Genetic Mutation','Heart Disease History','Hypertension','Inflammatory Bowel Disease','Smoker','Gender_M','Insurance Status_Uninsured','Urban or Rural_Urban','Treatment Type_Combination','Treatment Type_Radiotherapy','Treatment Type_Surgery','Family History and Genetic Mutation Interaction','Smoker and Alcohol Consumption']

categorical_columns = ['Cancer Stage','Country enc','Diet Risk','Healthcare Access','Insurance Costs','Obesity BMI','Physical Activity','Screening History','Age_groups','Tumor Size Categories','Comorbidity Count']

numerical_columns = ['Healthcare Costs','Incidence Rate per 100K','Mortality Rate per 100K','Tumor Size (mm)','Age','Mortality_to_Incidence_Ratio_per_100k']

datasets_names = [data_correlations]

### Tranform boolean to 0 and 1
columns = ['Survival Prediction', 'Gender_M','Insurance Status_Uninsured','Urban or Rural_Urban','Treatment Type_Combination','Treatment Type_Radiotherapy','Treatment Type_Surgery','Smoker and Alcohol Consumption','Family History and Genetic Mutation Interaction']

for i in datasets_names :
    for col in columns:
        i[col] = i[col].astype(int)

#Remove columns with ID and Date_Time
for dataset in datasets_names:
    dataset.drop('Date of Birth', axis=1, inplace=True)
    dataset.drop('Country', axis=1, inplace=True)


### Pearson Correlation: For two continuous variables.

In [ ]:
# Correlations Between Numeric Variables
selected_columns = data_correlations.loc[:, data_correlations.columns.isin(numerical_columns)]

corr_matrix = round(selected_columns.corr(method="pearson"),3)
mask = np.triu(np.ones_like(corr_matrix))
plt.figure(figsize=(16, 6))
heatmap = sns.heatmap(
    corr_matrix,
    mask=mask,
    vmin=-1,
    vmax=1,
    annot=True,
    cmap='coolwarm')
heatmap.set_title('Correlation Heatmap - Pearson', fontdict={'fontsize':12}, pad=12)
plt.show()

### Cramer's V: For two categorical variables.

In [ ]:
#Cramers V Categorical x Categorical
def cramerV(label, x):
  confusion_matrix = pd.crosstab(label, x)
  chi2 = chi2_contingency(confusion_matrix)[0]
  n = confusion_matrix.sum().sum()
  r, k = confusion_matrix.shape
  phi2 = chi2 / n
  phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
  rcorr = r - ((r - 1) ** 2) / (n - 1)
  kcorr = k - ((k - 1) ** 2) / (n - 1)

  try:
      if min((kcorr - 1), (rcorr - 1)) == 0:
          warnings.warn("Unable to calculate Cramer's V using bias correction. Consider not using bias correction", RuntimeWarning)
          v = 0
      else:
          v = np.sqrt(phi2corr / min((kcorr - 1), (rcorr - 1)))
  except:
      print("Error in Cramer's V calculation")
      v = 0
  return v

def plot_cramer(df):
  cramer = pd.DataFrame(index=df.columns, columns=df.columns, dtype=float)
  for column_of_interest in df.columns:
      for column in df.columns:
          if column_of_interest != column:
              v = cramerV(df[column_of_interest], df[column])
              cramer.loc[column_of_interest, column] = v

  cramer.fillna(value=0, inplace=True)

  # Mask the upper triangle
  mask = np.triu(np.ones_like(cramer, dtype=bool))

  plt.figure(figsize=(24, 24))
  sns.heatmap(
      cramer,
      mask=mask,
      annot=True,
      fmt=".3f",
      cmap="coolwarm",
      square=True,
      cbar_kws={"shrink": .8},
      vmin=0,  # Minimum value for the color scale
      vmax=1,    # Maximum value for the color scale
      linewidths=0,  # Remove gridlines between cells
      linecolor='none'  # Removes lines around cells
  )
  plt.title("Cramer's V Correlation Heatmap")
  plt.savefig('Cramer_V_Correlation_Heatmap.png', bbox_inches='tight')
  #plt.show() # ----> Remove "#" to make it work

#selected_columns = data_correlations.loc[:, data_correlations.columns.isin(boolean_columns)]
#plot_cramer(selected_columns)

#selected_columns = data_correlations.loc[:, data_correlations.columns.isin(categorical_columns)]
#plot_cramer(selected_columns)

selected_columns = data_correlations.loc[:, data_correlations.columns.isin(categorical_columns + boolean_columns)]
plot_cramer(selected_columns)


### One Way ANOVA and Point-Biserial  binary+continuous

In [ ]:
# Categorical X Numerical

#One Way ANOVA
def anova_explorer(df, numeric_vars, categorical_vars, alpha=0.05):
    for num_var in numeric_vars:
        for cat_var in categorical_vars:
            # Drop rows with missing data
            sub_df = df[[num_var, cat_var]].dropna()

            # Group numeric variable by each category
            groups = [sub_df[num_var][sub_df[cat_var] == group] for group in sub_df[cat_var].unique()]

            if len(groups) < 2:
                continue  # Skip if not enough groups to compare

            # Perform one-way ANOVA
            f_stat, p_val = stats.f_oneway(*groups)

            if p_val < alpha:
                print(f"\nSignificant result for '{num_var}' vs '{cat_var}' (p = {p_val:.4f})")

                # Descriptive stats
                print(f"{'Category':<20} {'Mean':>10} {'Variance':>15} {'Sample Size':>15}")
                for group_name, group_data in sub_df.groupby(cat_var):
                    mean = group_data[num_var].mean()
                    var = group_data[num_var].var()
                    n = group_data[num_var].count()
                    print(f"{str(group_name):<20} {mean:10.2f} {var:15.2f} {n:15}")

                # Violin plot
                plt.figure(figsize=(8, 5))
                sns.violinplot(x=cat_var, y=num_var, data=sub_df)
                plt.title(f"Violin Plot: {num_var} by {cat_var}")
                plt.tight_layout()
                plt.show()

#anova_explorer(data_correlations, numerical_columns, categorical_columns)

#Validate/Convert all boolean columns as Boolean
for col in boolean_columns:
    data_correlations[col] = data_correlations[col].astype(bool)

def calculate_point_biserial(df, boolean_columns, numeric_cols):
  correlation_matrix = pd.DataFrame(index=boolean_columns, columns=numeric_cols)

  for binary_col in boolean_columns:
      for numeric_col in numeric_cols:
          correlation, _ = pointbiserialr(df[binary_col], df[numeric_col])
          correlation_matrix.loc[binary_col, numeric_col] = correlation
  return correlation_matrix.astype(float)

correlation_matrix = calculate_point_biserial(data_correlations, boolean_columns, numerical_columns)

# Plot Point-Biserial Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f',
      vmin=0,  # Minimum value for the color scale
      vmax=1)   # Maximum value for the color scale
plt.title('Point-Biserial Correlation Heatmap')
plt.show()

## Data Normalization
Guide from https://www.geeksforgeeks.org/how-to-standardize-data-in-a-pandas-dataframe/


Chosen method -> Z-score, how many standar deviations away is from the mean.

In [ ]:
numerical_columns = ['Healthcare Costs','Incidence Rate per 100K','Mortality Rate per 100K','Tumor Size (mm)','Mortality_to_Incidence_Ratio_per_100k', 'Age']

# Fit scaler only on training data's numerical columns
scaler = StandardScaler()
scaler.fit(X_train_copy[numerical_columns])

# Transform only numerical columns, keep others unchanged
def scale_dataset(X, scaler, list_columns):
    X_scaled = X.copy()
    X_scaled[list_columns] = scaler.transform(X[list_columns])
    return X_scaled

X_train_copy_scaled = scale_dataset(X_train_copy, scaler, numerical_columns)
X_val_copy_scaled   = scale_dataset(X_val_copy, scaler, numerical_columns)
X_test_copy_scaled  = scale_dataset(X_test_copy, scaler, numerical_columns)

## Feature Selection

<strong> Notes on Feature Selection: </strong>

> 'Comorbidity Count' ---> Is a nice add on to the models

> 'Age_groups', 'Tumor Size Categories' ---> Can be used to replace 'Age' or 'Tumor Size (mm)'

> 'Family History and Genetic Mutation Interaction' ---> Can be used to replace 'Family History' and 'Genetic Mutation'

> 'Smoker and Alcohol Consumption' --> Can be used to replace 'Alcohol Consumption' and 'Smoker'


In [ ]:
X_train_copy_scaled.info()

In [ ]:
# List of boolean columns to convert
bool_columns = ['Gender_M','Insurance Status_Uninsured','Urban or Rural_Urban','Treatment Type_Combination','Treatment Type_Radiotherapy','Treatment Type_Surgery','Smoker and Alcohol Consumption', 'Family History and Genetic Mutation Interaction']

for df in [X_train_copy_scaled, X_val_copy_scaled, X_test_copy_scaled]:
    for col in bool_columns:
        df[col] = df[col].astype(int).astype('category')

#Remove columns with Date of Birth and Country
for df in [X_train_copy_scaled, X_val_copy_scaled, X_test_copy_scaled]:
    df.drop('Date of Birth', axis=1, inplace=True)
    df.drop('Country', axis=1, inplace=True)

categorical_columns_m = ['Cancer Stage','Country enc','Diet Risk','Healthcare Access','Insurance Costs','Obesity BMI','Physical Activity','Screening History', 'Alcohol Consumption','Diabetes','Early Detection','Family History','Genetic Mutation','Heart Disease History','Hypertension','Inflammatory Bowel Disease','Smoker','Gender_M','Insurance Status_Uninsured','Urban or Rural_Urban','Treatment Type_Combination','Treatment Type_Radiotherapy','Treatment Type_Surgery'
                         # If needed add columns after this comment

                         ]

numerical_columns_m = ['Healthcare Costs','Incidence Rate per 100K','Mortality Rate per 100K','Tumor Size (mm)'
                       # If needed add columns after this comment

                       ]

possible_add_columns = ['Age', "Age_groups","Tumor Size Categories",'Family History and Genetic Mutation Interaction','Smoker and Alcohol Consumption','Mortality_to_Incidence_Ratio_per_100k','Comorbidity Count']

#Guarantee that All categorical columns are categorical
for df in [X_train_copy_scaled, X_val_copy_scaled, X_test_copy_scaled]:
    for col in categorical_columns_m:
        df[col] = df[col].astype('category')

X_train_all = X_train_copy_scaled[categorical_columns_m + numerical_columns_m + possible_add_columns]
X_val_all = X_val_copy_scaled[categorical_columns_m + numerical_columns_m + possible_add_columns]
X_test_all = X_test_copy_scaled[categorical_columns_m + numerical_columns_m + possible_add_columns]

# Processed dataframes with original columns
X_train_processed = X_train_copy_scaled[categorical_columns_m+numerical_columns_m]
X_val_processed  = X_val_copy_scaled[categorical_columns_m+numerical_columns_m]
X_test_processed  = X_test_copy_scaled[categorical_columns_m+numerical_columns_m]

y_train_processed = y_train_copy
y_val_processed  = y_val_copy
y_test_processed  =y_test_copy

#Now all datasets are ready to next step

### RFE ANALYSIS

In [ ]:
# Separate the target variable
target_variable = y_train_processed

# Drop the target variable from the feature set
data_rfe = X_train_processed

model = LogisticRegression()
rfe = RFE(estimator=model, n_features_to_select=10)
X_rfe = rfe.fit_transform(X=data_rfe, y=target_variable)
selected_features = pd.Series(rfe.support_, index=data_rfe.columns)
#print(selected_features)

# Create a list of selected (True) columns
selected_columns = selected_features[selected_features == True].index.tolist()
print("List of selected features:", selected_columns)
X_train_rfe = X_train_processed.loc[:, selected_columns]
X_val_rfe = X_val_processed.loc[:, selected_columns]

y_train_rfe = y_train_processed[X_train_rfe.index]
y_val_rfe = y_val_processed[X_val_rfe.index]

# Data Modeling

- X_train_processed
- X_val_processed
- X_test_processed
- y_train_processed
- y_val_processed
- y_test_processed

### Baseline Model - OneRule Algorithm

As a benchmark, we want to develop a very simple Baseline Model at first. Like this we can then assess the performance of more sophisticated models compared to this benchmark.

We decided to use a OneRule Algorithm as our Baseline Model

In [ ]:
# First we discretize our variables in order to use them for the OneRule Algorithm

# Discretize into 5 quantile-based bins
discretizer = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')

# Fit only on the training set
X_train_discretized = pd.DataFrame(
    discretizer.fit_transform(X_train_rfe),
    columns=X_train_rfe.columns,
    index=X_train_rfe.index
).astype(int)

# Transform validation set using same bin edges
X_val_discretized = pd.DataFrame(
    discretizer.transform(X_val_rfe),
    columns=X_val_rfe.columns,
    index=X_val_rfe.index
).astype(int)


In [ ]:
# ---------------------------
# Definint a function for the OneR Classifier which we can later call on our data
# ---------------------------
class OneRClassifier:
    def fit(self, X, y):
        self.best_feature = None
        self.best_rule = {}
        self.default_class = y.mode()[0]
        min_error = float('inf')

        for column in X.columns:
            rules = {}
            total_error = 0
            for val in X[column].unique():
                subset = y[X[column] == val]
                most_common = subset.mode()[0]
                rules[val] = most_common
                total_error += (subset != most_common).sum()
            if total_error < min_error:
                min_error = total_error
                self.best_feature = column
                self.best_rule = rules

    def predict(self, X):
        predictions = []
        for _, row in X.iterrows():
            val = row[self.best_feature]
            pred = self.best_rule.get(val, self.default_class)
            predictions.append(pred)
        return np.array(predictions)


# OneR Model - Calling the function
oner = OneRClassifier()
oner.fit(X_train_discretized, y_train_rfe)

# Predictions
y_train_pred = oner.predict(X_train_discretized)
y_val_pred = oner.predict(X_val_discretized)

# Metrics Function
def get_metrics(y_true, y_pred, average='binary'):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC AUC': roc_auc_score(y_true, y_pred)}

# Evaluate
train_metrics = get_metrics(y_train_rfe, y_train_pred)
val_metrics = get_metrics(y_val_rfe, y_val_pred)

# Display as Table
results = pd.DataFrame({'Train': train_metrics, 'Validation': val_metrics})
print("OneR Classifier Performance:\n")
print(results.round(4))

In [ ]:
# Let's look at the confusion matrix to understand better the distribution of the predicted classes

print(confusion_matrix(y_val_rfe, y_val_pred))

Looking at the performance metrics and the confusion matrix, we can see a somehow strange and unexpected pattern here. The Recall value of 1.0 seems very unnatural and also the confusion matrix indicates that there is something wrong, as the model always predicts class 1 on the validation set, even if the dataset is not very unbalanced.

###Logistic Regression

After our Baseline Model, we want to use another simple algorithm to see how the results and the performance change compared to the OneRule Algorithm. For this, we chose to use Logistic Regression.

In [ ]:
# Train logistic regression
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_rfe, y_train_rfe)

# Predict
y_train_pred = logreg.predict(X_train_rfe)  #You cannot predict the values from the table used in training
y_val_pred = logreg.predict(X_val_rfe)

# Predict probabilities for ROC AUC
y_train_prob = logreg.predict_proba(X_train_rfe)[:, 1]
y_val_prob = logreg.predict_proba(X_val_rfe)[:, 1]

# Compute metrics
metrics = {
    "Accuracy": [
        accuracy_score(y_train_rfe, y_train_pred),
        accuracy_score(y_val_rfe, y_val_pred)
    ],
    "Precision": [
        precision_score(y_train_rfe, y_train_pred),
        precision_score(y_val_rfe, y_val_pred)
    ],
    "Recall": [
        recall_score(y_train_rfe, y_train_pred),
        recall_score(y_val_rfe, y_val_pred)
    ],
    "F1 Score": [
        f1_score(y_train_rfe, y_train_pred),
        f1_score(y_val_rfe, y_val_pred)
    ],
    "ROC AUC": [
        roc_auc_score(y_train_rfe, y_train_prob),
        roc_auc_score(y_val_rfe, y_val_prob)
    ]}

# Create a DataFrame
results_df = pd.DataFrame(metrics, index=["Train", "Validation"])
print(results_df.round(4))

In [ ]:
# Let's look at the confusion matrix to understand better the distribution of the predicted classes

print(confusion_matrix(y_val_rfe, y_val_pred))

Against our expectations, the Logistic Regression performs the same way as the OneRule Algorithm. The confusion matrix shows the same problem of predicting everything as class 1 and the performance metrics are almost identical, only the ROC AUC curve shows some minor improvement compared to our Baseline Model.

###Naive Bayes

After receiving somewhat dissatisfying results from our Baseline Model and the Logistic Regression, we will dive into probabilistic classifiers and train a Naive Bayes Algorithm on our data.

In [ ]:
# We will use the GaussianNB variant, as the dataset includes continuous, categorical and binary data and is scaled

# Initialize and train Naive Bayes
nb = GaussianNB()
nb.fit(X_train_rfe, y_train_rfe)

# Predict labels and probabilities
y_train_pred = nb.predict(X_train_rfe)
y_val_pred = nb.predict(X_val_rfe)

y_train_prob = nb.predict_proba(X_train_rfe)[:, 1]
y_val_prob = nb.predict_proba(X_val_rfe)[:, 1]

# Calculate metrics
metrics_nb = {
    "Accuracy": [
        accuracy_score(y_train_rfe, y_train_pred),
        accuracy_score(y_val_rfe, y_val_pred)
    ],
    "Precision": [
        precision_score(y_train_rfe, y_train_pred),
        precision_score(y_val_rfe, y_val_pred)
    ],
    "Recall": [
        recall_score(y_train_rfe, y_train_pred),
        recall_score(y_val_rfe, y_val_pred)
    ],
    "F1 Score": [
        f1_score(y_train_rfe, y_train_pred),
        f1_score(y_val_rfe, y_val_pred)
    ],
    "ROC AUC": [
        roc_auc_score(y_train_rfe, y_train_prob),
        roc_auc_score(y_val_rfe, y_val_prob)
    ]
}

# Create and display the metrics table
results_nb = pd.DataFrame(metrics_nb, index=["Train", "Validation"])
print(results_nb.round(4))


In [ ]:
# Let's look at the confusion matrix to understand better the distribution of the predicted classes

print(confusion_matrix(y_val_rfe, y_val_pred))

Even after the third model, we see the same pattern continuing. Again the model predicts every instance as class 1 and all metrics are the same as before besides the ROC AUC curve (which again slightly improved compared to the Logistic Regression).

After training 3 models, we are getting the notion that there might be something wrong with the preprocessing of our data or about the dataset itself.

## Instance Based

---

Across the instance Baseds Model we will go with the Ball Tree, which is suitable for dataset with an elevate number of features.

But first, we will evaluate how many neighbours are the optimum for our model based on the F1 Score and ROC AUC.

In [ ]:
#Using RFE features for better time efficiency
#This code can take up to 20 minutes

# Range of k values
numberK_list = np.arange(1, 31)

# Tracking best score
high_score = 0
best_k = 0
score_list_train = []
score_list_val = []

# Loop through k values
for k in numberK_list:
    model = KNeighborsClassifier(n_neighbors=k, algorithm='ball_tree')
    model.fit(X_train_rfe, y_train_rfe)

    # Get predicted probabilities for class 1
    train_probs = model.predict_proba(X_train_rfe)[:, 1]
    val_probs = model.predict_proba(X_val_rfe)[:, 1]

    # Compute AUC scores
    auc_train = roc_auc_score(y_train_rfe, train_probs)
    auc_val = roc_auc_score(y_val_rfe, val_probs)

    score_list_train.append(auc_train)
    score_list_val.append(auc_val)

    if auc_val > high_score:
        high_score = auc_val
        best_k = k

# Print best result
print(f"Best number of neighbors: {best_k}")
print(f"ROC AUC on train with {best_k} neighbors: {score_list_train[best_k - 1]:.4f}")
print(f"ROC AUC on validation with {best_k} neighbors: {high_score:.4f}")

# Plotting
plt.figure(figsize=(10, 5))
plt.plot(numberK_list, score_list_train, label='Train ROC AUC')
plt.plot(numberK_list, score_list_val, label='Validation ROC AUC')
plt.vlines(x=best_k, ymin=min(score_list_val), ymax=high_score, colors='green', linestyles='--', label='Best k')
plt.xticks(np.arange(1, 31, 2))
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('ROC AUC Score')
plt.title('ROC AUC vs. k (Ball Tree KNN with RFE features)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#Using RFE features for better time efficiency
#This code can take up to 20 minutes

numberK_list = np.arange(1, 31)
high_score = 0
best_k = 0
score_list_train = []
score_list_val = []

for k in numberK_list:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_rfe, y_train_rfe)

    # Predictions
    labels_train = model.predict(X_train_rfe)
    labels_val = model.predict(X_val_rfe)

    # F1 scores
    score_train = f1_score(y_train_rfe, labels_train, pos_label=1)
    score_val = f1_score(y_val_rfe, labels_val, pos_label=1)

    score_list_train.append(score_train)
    score_list_val.append(score_val)

    # Save best k
    if score_val > high_score:
        high_score = score_val
        best_k = k

# Results
print(f"Best number of neighbors: {best_k}")
print(f"Mean F1-score in train with {best_k} neighbors: {score_list_train[best_k - 1]:.4f}")
print(f"Mean F1-score in validation with {best_k} neighbors: {high_score:.4f}")


In [ ]:
# Initialize KNN using Ball Tree algorithm
knn_bt = KNeighborsClassifier(algorithm='ball_tree', n_neighbors=29)
knn_bt.fit(X_train_rfe, y_train_processed)

# Predict class labels
y_train_pred_bt = knn_bt.predict(X_train_rfe)
y_val_pred_bt = knn_bt.predict(X_val_rfe)

# Predict probabilities for ROC AUC
y_train_prob_bt = knn_bt.predict_proba(X_train_rfe)[:, 1]
y_val_prob_bt = knn_bt.predict_proba(X_val_rfe)[:, 1]

# Compute evaluation metrics
metrics_bt = {
    "Accuracy": [
        accuracy_score(y_train_processed, y_train_pred_bt),
        accuracy_score(y_val_processed, y_val_pred_bt)
    ],
    "Precision": [
        precision_score(y_train_processed, y_train_pred_bt),
        precision_score(y_val_processed, y_val_pred_bt)
    ],
    "Recall": [
        recall_score(y_train_processed, y_train_pred_bt),
        recall_score(y_val_processed, y_val_pred_bt)
    ],
    "F1 Score": [
        f1_score(y_train_processed, y_train_pred_bt),
        f1_score(y_val_processed, y_val_pred_bt)
    ],
    "ROC AUC": [
        roc_auc_score(y_train_processed, y_train_prob_bt),
        roc_auc_score(y_val_processed, y_val_prob_bt)
    ]
}

# Display results in a DataFrame
results_bt = pd.DataFrame(metrics_bt, index=["Train", "Validation"])
print("Ball Tree Classifier Performance:\n")
print(results_bt.round(4))

In [ ]:
# Let's look at the confusion matrix to understand better the distribution of the predicted classes

print(confusion_matrix(y_val_processed, y_val_pred_bt))

In [ ]:
# Initialize KNN using Ball Tree algorithm
knn_bt = KNeighborsClassifier(algorithm='ball_tree', n_neighbors=12)
knn_bt.fit(X_train_rfe, y_train_processed)

# Predict class labels
y_train_pred_bt = knn_bt.predict(X_train_rfe)
y_val_pred_bt = knn_bt.predict(X_val_rfe)

# Predict probabilities for ROC AUC
y_train_prob_bt = knn_bt.predict_proba(X_train_rfe)[:, 1]
y_val_prob_bt = knn_bt.predict_proba(X_val_rfe)[:, 1]

# Compute evaluation metrics
metrics_bt = {
    "Accuracy": [
        accuracy_score(y_train_processed, y_train_pred_bt),
        accuracy_score(y_val_processed, y_val_pred_bt)
    ],
    "Precision": [
        precision_score(y_train_processed, y_train_pred_bt),
        precision_score(y_val_processed, y_val_pred_bt)
    ],
    "Recall": [
        recall_score(y_train_processed, y_train_pred_bt),
        recall_score(y_val_processed, y_val_pred_bt)
    ],
    "F1 Score": [
        f1_score(y_train_processed, y_train_pred_bt),
        f1_score(y_val_processed, y_val_pred_bt)
    ],
    "ROC AUC": [
        roc_auc_score(y_train_processed, y_train_prob_bt),
        roc_auc_score(y_val_processed, y_val_prob_bt)
    ]
}

# Display results in a DataFrame
results_bt = pd.DataFrame(metrics_bt, index=["Train", "Validation"])
print("Ball Tree Classifier Performance:\n")
print(results_bt.round(4))

In [ ]:
# Let's look at the new confusion matrix to understand better the distribution of the predicted classes

print(confusion_matrix(y_val_processed, y_val_pred_bt))

From the results above, when we have 29 neighbours the model is not able to overcome a Random Model, although having the best F1 score. However as expected, with 13 neighbors the model is slightly better than a random, where the validation set is perfoming very above the curve (50.4%).

Interstingly, this model does not predicted "1" for all instnaces, unlike the previous models. Although as already stated it did not show any improvements on performance and predictability.

## Classification Tree

---

From the results above, when we have 29 neighbours the model is not able to overcome a Random Model, although having the best F1 score. However as expected, with 13 neighbors the model is slightly better than a random, where the validation set is perfoming very above the curve (50.4%).

Interstingly, this model does not predicted "1" for all instnaces, unlike the previous models. Although as already stated it did not show any improvements on performance and predictability.

**Gini impurity**

In [ ]:
modelDT = DecisionTreeClassifier()

In [ ]:
modelDT.fit(X_train_processed, y_train_processed)

In [ ]:
# Process a decision tree graph using Graphviz
from sklearn import tree

dot_data = tree.export_graphviz(modelDT,
                                out_file=None,
                                feature_names=X_train_processed.columns,
                                class_names=["Not Survived", "Survived"],
                                filled=True,
                                rounded=True,
                                special_characters=True)

graph = graphviz.Source(dot_data)
graph

Predicting the target variable based on the decision tree and later comparing the results to the actual values of the target variable.

In [ ]:
y_pred = modelDT.predict(X_val_processed)

In [ ]:
modelDT.score(X_train_processed, y_train_processed)

In [ ]:
modelDT.score(X_val_processed, y_val_processed)

As expected, the model is overfitting on the training data and on the test set the results are barely above 50%.



---



**Entropy based**

In [ ]:
modelDT_entropy = DecisionTreeClassifier(criterion='entropy')
modelDT_entropy.fit(X_train_processed, y_train_processed)

In [ ]:
modelDT.feature_importances_

In [ ]:
modelDT_entropy.score(X_train_processed, y_train_processed)

In [ ]:
modelDT_entropy.score(X_val_processed, y_val_processed)

In either cases, the model is overfitting on the training model and the score on the test set is disappointing. A deeper focus on the importance of each feature and pruning are required.



---



In the next section we will rank the features by importance for the decision tree model.

In [ ]:
def plot_feature_importances(model,DF):
    n_features = DF.shape[1]
    plt.figure(figsize=(5,3))
    plt.barh(range(n_features), model.feature_importances_, color='yellowgreen')
    plt.yticks(np.arange(n_features), DF.columns)
    plt.xlabel("Feature importance")
    plt.ylabel("Feature")
    plt.title('Feature Importance in Decision Tree Classifier')
    plt.show()

In [ ]:
# Get feature importances
importances = modelDT.feature_importances_
indices = np.argsort(importances)

# Plot manually with smaller Y-axis labels
plt.figure(figsize=(10, 6))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], align="center")
plt.yticks(range(len(indices)), X_train_processed.columns[indices], fontsize=8)
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

The analysis above would suggest that the correct number of features used should be between 1 and 5.

**Pruning**

In [ ]:
modelDT_maxdepth4 = DecisionTreeClassifier(max_depth=4)
modelDT_maxdepth4.fit(X_train_processed, y_train_processed)
modelDT_maxdepth4.score(X_train_processed, y_train_processed)
modelDT_maxdepth4.score(X_val_processed, y_val_processed)

In [ ]:
modelDT_maxdepth2 = DecisionTreeClassifier(max_depth=2)
modelDT_maxdepth2.fit(X_train_processed, y_train_processed)
modelDT_maxdepth2.score(X_train_processed, y_train_processed)
modelDT_maxdepth2.score(X_val_processed, y_val_processed)

It seems as the least depth we use, the better model we get, even though the score is still low (60%).
We build a plot that shows the different scores and overfitting of the decision tree depending on pre-set maximum depth of the model.

In [ ]:
scores_train = []
scores_val = []
for i in range(1,9):
    DTC = DecisionTreeClassifier(max_depth=i)
    DTC.fit(X_train_processed, y_train_processed)
    scores_train.append(DTC.score(X_train_processed, y_train_processed))
    scores_val.append(DTC.score(X_val_processed, y_val_processed))

In [ ]:
plt.plot(list(range(1,9)), scores_train, label="Score on Training Set", color='yellowgreen')
plt.plot(list(range(1,9)), scores_val, label="Score on Validation Set", color='dimgray')
plt.xlabel("Maximum Depth")
plt.ylabel("Score")
plt.legend()
plt.show()

The graph compares the scores between training set and validation set. We can see that until the first three maximum depth levels, the results between the training and validation set are similar, whereas from 4 onwards the score start to increase in the training set - due to overfitting - and decrease on the validation set. The ideal cut seems indeed to be at maximum depth = 3.

Now we can add the maximum number of leaf nodes to the graph and see if this adds relevant information

In [ ]:
scores_train = []
scores_val = []
parameters = []
for i in range(10, 20):
    for j in range(1,7):
        parameters.append([i,j])
        DTR = DecisionTreeClassifier(min_samples_split=i, max_depth=j)
        DTR.fit(X_train_processed, y_train_processed)
        scores_train.append(DTR.score(X_train_processed, y_train_processed))
        scores_val.append(DTR.score(X_val_processed, y_val_processed))

In [ ]:
len(scores_train)

In [ ]:
scores = pd.DataFrame({'Score_Train': scores_train,'Score_Validation': scores_val,'Parameters': parameters}).sort_values(by=['Score_Train'])
plt.figure(figsize=(16,6))
plt.plot(list(range(1,61)), scores['Score_Train'], label="Score on Training Set", color='yellowgreen')
plt.plot(list(range(1,61)), scores['Score_Validation'], label="Score on Validation Set", color='dimgray')
plt.xlabel("Minimum Samples Required to Split, and Maximum Depth")
plt.ylabel("Score")
plt.xticks(np.arange(len(parameters)), scores['Parameters'])
plt.xticks(rotation=90)
plt.legend()
plt.show()

In [ ]:
modelDT_leafs2 = DecisionTreeClassifier(max_leaf_nodes=2)
modelDT_leafs2.fit(X_train_processed, y_train_processed)
modelDT_leafs2.score(X_train_processed, y_train_processed)
modelDT_leafs2.score(X_val_processed, y_val_processed)

Other than confirming that the optimal maximum length of the tree is 3, it also suggests that increasing the minimum samples required to split does not change the performance, so the value imputed to max_leaf_nodes is irrelevant to the final result.

In [ ]:
# 1. Select features
selected_features = [
    "Healthcare Costs",
    "Incidence Rate per 100K",
    "Tumor Size (mm)",
    "Mortality Rate per 100K",
    "Country enc"
]

X_selected_train = X_train_processed[selected_features]
X_selected_val = X_val_processed[selected_features]

# 2. Fit the decision tree
modelDT = DecisionTreeClassifier(
    max_depth=2,
    max_leaf_nodes=5,
    random_state=33
)

modelDT.fit(X_selected_val, y_val_processed)

# 3. Visualize the tree
dot_data = export_graphviz(
    modelDT,
    out_file=None,
    feature_names=selected_features,
    class_names=["Not Survived", "Survived"],
    filled=True,
    rounded=True,
    special_characters=True
)
graph = graphviz.Source(dot_data)
display(graph)

# 4. Predictions
y_pred_val = modelDT.predict(X_selected_val)
y_pred_train = modelDT.predict(X_selected_train)

# 5. Evaluation metrics
acc_train = accuracy_score(y_train_processed, y_pred_train)
rec_train = recall_score(y_train_processed, y_pred_train, pos_label=1)  # adjust label if needed
prec_train = precision_score(y_train_processed, y_pred_train, pos_label=1)
f1_train = f1_score(y_train_processed, y_pred_train, pos_label=1)
roc_auc_train = roc_auc_score(y_train_processed, y_pred_train)
acc_val = accuracy_score(y_val_processed, y_pred_val)
rec_val = recall_score(y_val_processed, y_pred_val, pos_label=1)  # adjust label if needed
prec_val = precision_score(y_val_processed, y_pred_val, pos_label=1)
f1_val = f1_score(y_val_processed, y_pred_val, pos_label=1)
roc_auc_val = roc_auc_score(y_val_processed, y_pred_val)
cm = confusion_matrix(y_val_processed, y_pred_val)

# 6. Print results
print(f"Accuracy Train:  {acc_train:.4f}")
print(f"Recall Train:    {rec_train:.4f}")
print(f"Precision Train: {prec_train:.4f}")
print(f"F1 Score Train:  {f1_train:.4f}")
print(f"ROC AUC Train:   {roc_auc_train:.4f}")
print(f"Accuracy Validation:  {acc_val:.4f}")
print(f"Recall Validation: {rec_val:.4f}")
print(f"Precision Validation: {prec_val:.4f}")
print(f"F1 Score Validation:  {f1_val:.4f}")
print(f"ROC AUC Validation:   {roc_auc_val:.4f}")
print("Confusion Matrix:")
print(cm)

# 7. Confusion matrix as heatmap
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Not Survived", "Survived"], yticklabels=["Not Survived", "Survived"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Validation Set")
plt.tight_layout()
plt.show()

Conclusion: given the nature of the data the best F1 score we can reach is 0.75 and this is done by predicting "Survived" on all or almost all patients. It seems as any kind of pruning does not affect the final score and that the simplest model in terms of nodes and leafs still allows to reach the same results, as we confirm below.

In [ ]:
# 1. Select features
selected_features = [
    "Healthcare Costs"
]

X_selected_train = X_train_processed[selected_features]
X_selected_val = X_val_processed[selected_features]

# 2. Fit the decision tree
modelDT = DecisionTreeClassifier(
    max_depth=1,
    max_leaf_nodes=2,
    random_state=33
)

modelDT.fit(X_selected_val, y_val_processed)

# 3. Visualize the tree
dot_data = export_graphviz(
    modelDT,
    out_file=None,
    feature_names=selected_features,
    class_names=["Not Survived", "Survived"],
    filled=True,
    rounded=True,
    special_characters=True
)
graph = graphviz.Source(dot_data)
display(graph)

# 4. Predictions
y_pred_val = modelDT.predict(X_selected_val)
y_pred_train = modelDT.predict(X_selected_train)

# 5. Evaluation metrics
acc_train = accuracy_score(y_train_processed, y_pred_train)
rec_train = recall_score(y_train_processed, y_pred_train, pos_label=1)  # adjust label if needed
prec_train = precision_score(y_train_processed, y_pred_train, pos_label=1)
f1_train = f1_score(y_train_processed, y_pred_train, pos_label=1)
acc_val = accuracy_score(y_val_processed, y_pred_val)
rec_val = recall_score(y_val_processed, y_pred_val, pos_label=1)  # adjust label if needed
prec_val = precision_score(y_val_processed, y_pred_val, pos_label=1)
f1_val = f1_score(y_val_processed, y_pred_val, pos_label=1)
cm = confusion_matrix(y_val_processed, y_pred_val)

# 6. Print results
print(f"Accuracy Train:  {acc_train:.4f}")
print(f"Recall Train:    {rec_train:.4f}")
print(f"Precision Train: {prec_train:.4f}")
print(f"F1 Score Train:  {f1_train:.4f}")
print(f"Accuracy Validation:  {acc_val:.4f}")
print(f"Recall Validation: {rec_val:.4f}")
print(f"Precision Validation: {prec_val:.4f}")
print(f"F1 Score Validation:  {f1_val:.4f}")
print("Confusion Matrix:")
print(cm)

# 7. Confusion matrix as heatmap
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Not Survived", "Survived"], yticklabels=["Not Survived", "Survived"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Validation Set")
plt.tight_layout()
plt.show()

##  Ensemble Classifiers & Neural Network

---

We will start the Ensemble Classifiers by training 1 simple model of each type to get some initial insights:

In [ ]:
# Random Forest
rf_clf = RandomForestClassifier()
rf_clf.fit(X_train_processed, y_train_processed)
y_pred_rf = rf_clf.predict(X_val_processed)
print("Completed RandomForestClassifier")

F1_metric_rf = f1_score(y_val_processed, y_pred_rf)
print(f"F1-Score (Random Forest): {F1_metric_rf:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_val_processed, y_pred_rf))
print("Classification Report:")
print(classification_report(y_val_processed, y_pred_rf))

importances_rf = rf_clf.feature_importances_
feature_imp_df_rf = pd.DataFrame({
    'Feature': X_train_processed.columns,
    'Importance': importances_rf
}).sort_values('Importance', ascending=False)
print(feature_imp_df_rf)

# Extra Trees
et_clf = ExtraTreesClassifier()
et_clf.fit(X_train_processed, y_train_processed)
y_pred_et = et_clf.predict(X_val_processed)
print("Completed ExtraTreesClassifier")

F1_metric_et = f1_score(y_val_processed, y_pred_et)
print(f"F1-Score (Extra Trees): {F1_metric_et:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_val_processed, y_pred_et))
print("Classification Report:")
print(classification_report(y_val_processed, y_pred_et))

importances_et = et_clf.feature_importances_
feature_imp_df_et = pd.DataFrame({
    'Feature': X_train_processed.columns,
    'Importance': importances_et
}).sort_values('Importance', ascending=False)
print(feature_imp_df_et)

# AdaBoost
adaboost_clf = AdaBoostClassifier()
adaboost_clf.fit(X_train_processed, y_train_processed)
y_pred_ab = adaboost_clf.predict(X_val_processed)
print("Completed AdaBoostClassifier")

F1_metric_ab = f1_score(y_val_processed, y_pred_ab)
print(f"F1-Score (AdaBoost): {F1_metric_ab:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_val_processed, y_pred_ab))
print("Classification Report:")
print(classification_report(y_val_processed, y_pred_ab))

importances_ab = adaboost_clf.feature_importances_
feature_imp_df_ab = pd.DataFrame({
    'Feature': X_train_processed.columns,
    'Importance': importances_ab
}).sort_values('Importance', ascending=False)
print(feature_imp_df_ab)

# Gradient Boosting
gb_clf = GradientBoostingClassifier()
gb_clf.fit(X_train_processed, y_train_processed)
y_pred_gb = gb_clf.predict(X_val_processed)
print("Completed GradientBoostingClassifier")

F1_metric_gb = f1_score(y_val_processed, y_pred_gb)
print(f"F1-Score (Gradient Boosting): {F1_metric_gb:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_val_processed, y_pred_gb))
print("Classification Report:")
print(classification_report(y_val_processed, y_pred_gb))

importances_gb = gb_clf.feature_importances_
feature_imp_df_gb = pd.DataFrame({
    'Feature': X_train_processed.columns,
    'Importance': importances_gb
}).sort_values('Importance', ascending=False)
print(feature_imp_df_gb)


From the initial development of Ensemble Classifiers we got a few initial insights to develop further with hyperparamether tuning:
- Random Forest and Extra Trees: Both models are heavily biased toward class 1, rarely identifying any class 0 correctly. Possible reasons include class imbalance or uninformative features. For further development, we should address this issue by working to reduce uninformative features and, where possible, balance the data.
- AdaBoostClassifier & GradientBoostingClassifier: Again, we observe that both models predict nearly all samples as class 1, effectively ignoring class 0. Additionally, our experiments show that if we try to optimize these models for class 0 recall, they end up predicting everything as class 0.
- Final Conclusion: Our main takeaway is the need to continue further development, aiming to improve prediction quality for the non-dominant class while maintaining good performance on the dominant class.

Based on previous discoverys, we opted to look for the best set of hyperparameters with balanced_accuracy and weighted f1-score in mind.

### Hyperparameter tuning function

Here, we define the function used to search for the best set of hyperparameters for all Ensemble Classifiers and the Neural Network. We use Stratified K-Folds, optimized for balanced accuracy. This choice aligns with our goal of addressing the problem of imbalanced data. In this way, we can train the model without undersampling, thus avoiding potential information loss.

In [ ]:
def hyperparameter_tune(base_model, parameters, n_iter, kfold, X, y, seed=1234):
    start_time = time.time()

    k = StratifiedKFold(n_splits=kfold, shuffle=True, random_state=seed)

    scorer = make_scorer(balanced_accuracy_score)

    optimal_model = RandomizedSearchCV(
        base_model,
        param_distributions=parameters,
        n_iter=n_iter,
        cv=k,
        n_jobs=-1,
        random_state=seed,
        scoring=scorer)

    optimal_model.fit(X, y)

    stop_time = time.time()
    mean_score = optimal_model.best_score_
    std_score = optimal_model.cv_results_['std_test_score'][optimal_model.best_index_]

    print("Elapsed Time:", time.strftime("%H:%M:%S", time.gmtime(stop_time - start_time)))
    print("====================")
    print("Cross Val Mean: {:.3f}, Cross Val Stdev: {:.3f}".format(mean_score, std_score))
    print("Best Score: {:.3f}".format(optimal_model.best_score_))
    print("Best Parameters: {}".format(optimal_model.best_params_))
    return optimal_model.best_params_, optimal_model.best_score_

On the previous chunk we defined the Hyperparameter Tuning Function (hyperparameter_tune) with a purpose of finding the best hyperparameters for our machine learning model using random search with cross-validation.
Key notes about our function:


*   We are using stratified K-folds to preserve the class distribution
*   We also use RandomizedSearchCV to sample random hyperparameter settings from parameters, optimizing for balanced accuracy (noting that we also did other experiments here, e.g. Weighted F1-score or accuracy).
*   The output here will be the elapsed time for each search, Mean and standard deviation of cross-validation scores and finally the best score and corresponding parameters.

The choice for randommized search CV was due to the fact that we noticed a lot of noise in our dataset. This function will be used for Random Forest, Extra Trees, Adaboost, Gradient Boosting and MLP Classifier.



### MLP CLASSIFIER (Neural Network)

Now, we start to implement the function previously definied to look for the best set of hyperparameters, starting with MLP Classifier.

We defined 3 as the number of folds and 60 for iterations because these were the values recommended to us for being the most time-effective combination. We used this combination for all Ensemble Classifiers & neural network.

In [ ]:
scores = []
folds = [3]

for i in folds:
    print("\ncv = ", i)

    parameters = {
        "hidden_layer_sizes": [(50,), (100,), (50, 50), (100, 50)],
        "activation": ["tanh", "relu"],
        "solver": ["adam", "sgd"],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate": ["constant", "adaptive"],
        'max_iter':[800, 1000, 1200]}

    base_model = MLPClassifier(max_iter=500, random_state=1234)

    best_params, best_score = hyperparameter_tune(
        base_model, parameters, 60, i, X_train_processed, y_train_processed)
    scores.append(best_score)

# Train final model with best parameters
mlp_clf = MLPClassifier(**best_params, random_state=1234)
mlp_clf.fit(X_train_processed, y_train_processed)


for name, X, y_true in [
    ("Validation", X_val_processed, y_val_processed),
    ("Training", X_train_processed, y_train_processed)]:

    y_pred = mlp_clf.predict(X)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    conf_matrix = confusion_matrix(y_true, y_pred)

    print(f"\n=== {name} Set Results ===")
    print("Confusion Matrix:")
    print(conf_matrix)
    print(f"F1-score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print("Classification Report:")
    print(classification_report(y_true, y_pred))

print("\nBest cross-validated score during tuning:", best_score)


Here, we did the training of the MLP classifier. We defined 3 as the number of folds, to get the best value within a limited amount of time. The chosen evaluation metrics were the F1-score, confusion matrix and a classification report.

---

- Total time spent tuning hyperparameters +/- 1 hour.
- The MLP Classifier did not fully converge after 500 epochs, to address this problem we increased the scope for a minimum of 800 epochs.
- The average balanced accuracy across cross-validation folds was +/- 0.504 (very low, considering that 0.5 is random guess for binary classification problems). With standard deviation also low, the model is consistently getting about 50% balanced accuracy.
- Dictionary of the best parameters found in the search:
  Best Parameters: `{solver': 'adam', 'max_iter': 800, 'learning_rate': 'constant', 'hidden_layer_sizes': (50, 50), 'alpha': 0.0001, 'activation': 'relu'}`


---

|               | **Predicted 0** | **Predicted 1** |
|---------------|:---------------:|:---------------:|
| **Actual 0**  |    **1274**     |    **3220**     |
| **Actual 1**  |    **1919**     |    **4840**     |

- **Model predicts class 1 much more often than class 0.**

---

- F1-score (harmonic mean of precision and recall) was 0.6448 indicating a moderate score.
    - Class 0: F1 = 0.35 (low; most class 0s are missed)
    - Class 1: F1 = 0.64 (better, but not great)
- Overall accuracy: 54%
- Macro avg: Being o.5 across all classes, this is not a good indicator, sinalizing the possible the existance of imbalance in the performance.
- Weighted avg: The values are indicating that our model is just sligtly better than random guessing, being also slighly better on recall (0.54) than on Precision (0.52)
---
- Final balanced accuracy score on cross-validation was 0.50427.



In [ ]:
X_train_processed_rf = X_train_processed[['Healthcare Costs','Tumor Size (mm)', 'Incidence Rate per 100K', 'Mortality Rate per 100K', 'Country enc']]
X_val_processed_rf = X_val_processed[['Healthcare Costs','Tumor Size (mm)', 'Incidence Rate per 100K', 'Mortality Rate per 100K', 'Country enc']]

### Random Forest

Here, we search for the best set of hyperparameters for the Random Forest classifier. The chosen set of parameters is aimed at addressing class imbalance. We iteratively adjusted the parameter list multiple times to achieve the best performance.

In [ ]:
scores = []
folds = [3]

for i in folds:
    print("\ncv = ", i)

    param_dist = {
    "n_estimators":  [400, 600, 800, 1000],
    "max_depth":     list(range(4, 13)) + [None],
    "min_samples_leaf": [8, 12, 16, 24, 32],
    "criterion":     ["gini", "entropy"],
    "bootstrap":     [True],
    "class_weight":  ["balanced", "balanced_subsample"],
    "max_features":  ["sqrt", "log2", 0.3] }

    base_model = RandomForestClassifier(n_jobs=-1, random_state=1234)

    best_params, best_score = hyperparameter_tune(
        base_model, param_dist, 60, i, X_train_processed_rf, y_train_processed)
    scores.append(best_score)

# Train final model on undersampled data
rf_clf = RandomForestClassifier(**best_params)
rf_clf.fit(X_train_processed_rf, y_train_processed)

for name, X, y_true in [
  ("Validation", X_val_processed_rf, y_val_processed),
  ("Training", X_train_processed_rf, y_train_processed)]:

    y_pred = rf_clf.predict(X)

    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    conf_matrix = confusion_matrix(y_true, y_pred)

    print(f"\n=== {name} Set Results ===")
    print("Confusion Matrix:")
    print(conf_matrix)
    print(f"F1-score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print("Classification Report:")
    print(classification_report(y_true, y_pred))
print("\nBest cross-validated score during tuning:", best_score)


Here, we did the training of the **Random Forest** classifier. We defined 3 as the number of folds, to get the best value within a limited amount of time. The chosen evaluation metrics were the F1-score, confusion matrix, and a classification report.

---

- **Total time spent tuning hyperparameters:** +/- 28 minutes.
- The Random Forest model achieved its best performance with 400 estimators, a maximum depth of 10, and class weights set to `balanced_subsample`.
- The average balanced accuracy across cross-validation folds was **+/- 0.511**,  close to random guessing for binary classification. Standard deviation is very low, showing performance is consistent across splits.
- **Dictionary with the best parameters found in the search:**
  `{''n_estimators': 400, 'min_samples_leaf': 24, 'max_features': 'sqrt', 'max_depth': 10, 'criterion': 'entropy', 'class_weight': 'balanced_subsample', 'bootstrap': True}`


---

|               | **Predicted 0** | **Predicted 1** |
|---------------|:---------------:|:---------------:|
| **Actual 0**  |    **1728**     |    **2766**     |
| **Actual 1**  |    **2481**     |    **4278**     |

- **Model predicts class 1 more often than class 0, but the bias is less pronounced compared to the MLP.**

---

- **F1-score was 0.6201, indicating a moderate performance.**
    - **Class 0:** F1 = 0.40 (still low; a significant portion of class 0 samples are missed)
    - **Class 1:** F1 = 0.62 (better, but still far from ideal)
- **Overall accuracy:** 54%
- **Macro avg:** 0.51 across all classes, which again suggests the model’s performance is only slightly above random and not balanced across both classes.
- **Weighted avg:** 0.54 accuracy, slightly favoring recall (0.54) over precision (0.53), showing a small improvement over random guessing.

---

- **Final balanced accuracy score on cross-validation was 0.511.**

---

- **Top 5 most important features according to Gini importance:**
    1. Healthcare Costs (0.28)
    2. Tumor Size (mm) (0.22)
    3. Incidence Rate per 100K (0.18)
    4. Mortality Rate per 100K (0.16)
    5. Country enc (0.15)

### Extra Trees

Extra Trees was an option due to the fact that random forest produced good results. This is another similar approach being the biggest difference between them the fact that extra trees use the entire dataset while random forest uses bootstrap samples.

In [ ]:
scores = []
folds = [3]

for i in folds:
    print("\ncv = ", i)

    parameters = {
        "max_depth": list(range(2, 10)),
        "n_estimators": [100, 200, 300, 400, 500],
        "criterion": ["gini", "entropy"],
        "min_samples_leaf": list(range(20, 45, 2)),
        "class_weight": ['balanced', 'balanced_subsample'] }

    base_model = ExtraTreesClassifier(n_jobs=-1, random_state=1234)

    best_params, best_score = hyperparameter_tune(
        base_model, parameters, 60, i, X_train_processed, y_train_processed)
    scores.append(best_score)

et_clf = ExtraTreesClassifier(**best_params)

et_clf.fit(X_train_processed, y_train_processed)
y_pred = et_clf.predict(X_val_processed)

# Evaluate the model
for name, X, y_true in [
  ("Validation", X_val_processed, y_val_processed),
  ("Training", X_train_processed, y_train_processed)]:

    y_pred = et_clf.predict(X)

    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    conf_matrix = confusion_matrix(y_true, y_pred)

    print(f"\n=== {name} Set Results ===")
    print("Confusion Matrix:")
    print(conf_matrix)
    print(f"F1-score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print("Classification Report:")
    print(classification_report(y_true, y_pred))
print("\nBest cross-validated score during tuning:", best_score)

importances = et_clf.feature_importances_
feature_imp_df = pd.DataFrame({'Feature': X_train_processed.columns, 'Importance': importances}).sort_values('Importance', ascending=False)
print(feature_imp_df)

Here, we did the training of the **Extra Trees** classifier. Again, we chose the metrics to be F1-score, confusion matrix, and a classification report.

---

- **Total time spent tuning hyperparameters:** +/- 7 minutes.
- The Random Forest model achieved its best performance with 400 estimators, a maximum depth of 8, and class weights set to `balanced_subsample`.
- The average balanced accuracy across cross-validation folds was **+/- 0.503** again close to random guessing for binary classification. Standard deviation is very low, showing performance is consistent across splits.
- **Dictionary of best parameters found in the search:**
  `{'n_estimators': 400, 'min_samples_leaf': 32, 'max_depth': 8, 'criterion': 'entropy', 'class_weight': 'balanced_subsample'}`
  - CHANGE TO 'n_estimators': 300, 'min_samples_leaf': 34, 'max_depth': 9, 'criterion': 'gini', 'class_weight': 'balanced'

---

|               |   **Predicted 0**   |   **Predicted 1**   |
|---------------|:------------------:|:------------------:|
| **Actual 0**  |     **1845**       |     **2649**       |
| **Actual 1**  |     **2662**       |     **4097**       |

- **Model predicts class 1 more often than class 0, and misses a large number of class 1 cases.**

---

- **F1-score (harmonic mean of precision and recall) was 0.6067, indicating moderate performance.**
    - **Class 0:** F1 = 0.41 (still low; a significant portion of class 0 samples are missed)
    - **Class 1:** F1 = 0.61 (better, but still far from ideal)
- **Overall accuracy:** 53%
- **Macro avg:** 0.51 across all classes, showing the model is just barely above random and not balanced across classes.
- **Weighted avg:** 0.53, with performance just above random guessing.

---

- **Final balanced accuracy score on cross-validation was 0.503.**

---

- **Top 5 most important features according to feature importance:**
    1. Country enc (0.0601)
    2. Insurance Costs (0.0454)
    3. Mortality Rate per 100K (0.0453)
    4. Incidence Rate per 100K (0.0444)
    5. Cancer Stage (0.0440)

### Adaboost

This is our first use of a boosting algorithm. Adaptive Boosting was the initial choice.

In [ ]:
scores = []
folds = [3]

for i in folds:
    print("\ncv = ", i)

    # Hyperparameters for AdaBoostClassifier
    parameters = {
        "n_estimators": [150, 200, 250],
        "learning_rate": [0.009, 0.01, 0.02]}

    base_model = AdaBoostClassifier(random_state=1234)

    best_params, best_score = hyperparameter_tune(
        base_model, parameters, 60, i, X_train_processed, y_train_processed)
    scores.append(best_score)

# Extract base estimator parameters
adaboost_clf = AdaBoostClassifier( n_estimators=best_params.get('n_estimators'),learning_rate=best_params.get('learning_rate'), random_state=1234 )
adaboost_clf.fit(X_train_processed, y_train_processed)

# Evaluate AdaBoost
for name, X, y_true in [
    ("Validation", X_val_processed, y_val_processed),
    ("Training", X_train_processed, y_train_processed)]:
    y_pred = adaboost_clf.predict(X)

    f1 = f1_score(y_true, y_pred, average='weighted')
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    conf_matrix = confusion_matrix(y_true, y_pred)

    print(f"\n=== {name} Set Results (AdaBoost) ===")
    print("Confusion Matrix:")
    print(conf_matrix)
    print(f"F1-score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print("Classification Report:")
    print(classification_report(y_true, y_pred))

print("\nBest cross-validated score during tuning:", best_score)

Here, we did the training of the **AdaBoost** classifier. We kept the same metrics defined previously.

---

- **Total time spent tuning hyperparameters:** +/- 1 minute.
- The AdaBoost model finished quickly, exploring all parameter combinations, however, the parameter space was smaller than previous models.
- The average balanced accuracy across cross-validation folds was **0.500**, that means that our model performance is similar to random predicting. Standard deviation was zero, indicating perfectly consistent performance.
- **Dictionary of best parameters found in the search:**
  `{'n_estimators': 150, 'learning_rate': 0.009}`

---

|               |   **Predicted 0**   |   **Predicted 1**   |
|---------------|:------------------:|:------------------:|
| **Actual 0**  |     **0**          |     **4494**       |
| **Actual 1**  |     **0**          |     **6759**       |

- **The model predicts only class 1 for all inputs (completely ignores class 0).**
---

- **F1-score (harmonic mean of precision and recall) was 0.7505, but this is misleading due to the complete absence of class 0 predictions.**

After this considerations, we will only highlight some other key metrics:

- **Macro avg:** 0.30 precision, 0.50 recall, 0.38 f1-score, we are in the presense of severe imbalance.
- **Weighted avg:** 0.36 precision, 0.60 recall, 0.45 f1-score, this results derive from the fact that our model is only predicting ones.
- **Final balanced accuracy score on cross-validation was 0.5, that is equivalent to random guess.**



### Gradient Boosting

Gradient Boosting is our final single model that we will look for the best set of parameters.

In [ ]:
scores = []
folds = [3]

for i in folds:
    print("\ncv = ", i)

    parameters = {
        "n_estimators": [50, 100, 150],
        "learning_rate": [0.005, 0.009, 0.01, 0.02],
        "max_depth": [4,6,8],
        "min_samples_split": [10, 15, 20],
        "min_samples_leaf": [1, 2, 4],
        "subsample": [0.8, 1.0]}

    base_model = GradientBoostingClassifier(random_state=1234)
    best_params, best_score = hyperparameter_tune(base_model, parameters, 60, i, X_train_processed, y_train_processed)
    scores.append(best_score)

gradient_boosting_clf = GradientBoostingClassifier(
    n_estimators=best_params.get('n_estimators'),
    learning_rate=best_params.get('learning_rate'),
    max_depth=best_params.get('max_depth'),
    min_samples_split=best_params.get('min_samples_split'),
    min_samples_leaf=best_params.get('min_samples_leaf'),
    subsample=best_params.get('subsample'),
    max_features=best_params.get('max_features'),
    random_state=1234)

gradient_boosting_clf.fit(X_train_processed, y_train_processed)

for name, X, y_true in [
    ("Validation", X_val_processed, y_val_processed),
    ("Training", X_train_processed, y_train_processed)
]:
    y_pred = gradient_boosting_clf.predict(X)

    f1 = f1_score(y_true, y_pred, average='weighted')
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    conf_matrix = confusion_matrix(y_true, y_pred)

    print(f"\n=== {name} Set Results (Gradient Boosting) ===")
    print("Confusion Matrix:")
    print(conf_matrix)
    print(f"F1-score: {f1:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print("Classification Report:")
    print(classification_report(y_true, y_pred))

print(f"\nBest score from hyperparameter tuning: {best_score:.4f}")

# Feature Importances
if hasattr(gradient_boosting_clf, "feature_importances_"):
    importances = gradient_boosting_clf.feature_importances_
    feature_imp_df = pd.DataFrame({
        'Feature': X_train_processed.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    print(feature_imp_df)
else:
    print("GradientBoostingClassifier does not provide feature importances.")

Here, we did the training of the **Gradient Boosting** classifier. Again, with the same metrics.

---

- **Total time spent tuning hyperparameters:** +/- 10 minutes.
- The average balanced accuracy across cross-validation folds was **0.501** (very close to what would be random guessing for binary classification). Standard deviation was very low, indicating consistent performance across splits.
- **Dictionary of best parameters found in the search:**
  `{'subsample': 1.0, 'n_estimators': 100, 'min_samples_split': 20, 'min_samples_leaf': 1, 'max_depth': 8, 'learning_rate': 0.02}`

---

|               |   **Predicted 0**   |   **Predicted 1**   |
|---------------|:------------------:|:------------------:|
| **Actual 0**  |     **29**         |     **4465**       |
| **Actual 1**  |     **46**         |     **6713**       |

- **The model predicts class 1 almost exclusively, missing almost all class 0 cases.**

---

The Gradient Boosting classifier, like previous model AdaBoost, struggled to provide meaningful discrimination between classes for this dataset. This indicates a lack of predictive signal in the current features.


## Model Stacking

---

After running all the previous models, we now chose what we considered to be the best performing models and performed a 2-level stacking solution using as base learners Random Forest, Extra Trees and MLP Classifier. Our initial meta learner is Logistic Regression, later on, we changed to Random Forest to achieve better performance.

In [ ]:
#------------------->>> We need to select manually the best parameters

start_time = time.time()
params_rf = {'n_estimators': 400, 'min_samples_leaf': 24, 'max_features': 'sqrt', 'max_depth': 10, 'criterion': 'entropy', 'class_weight': 'balanced_subsample', 'bootstrap': True, 'n_jobs':-1, 'random_state':1234}
rf_clf = RandomForestClassifier(**params_rf)
rf_clf.fit(X_train_processed, y_train_processed)
print("Completed RandomForestClassifier")
stop_time = time.time()
print("Elapsed Time:", time.strftime("%H:%M:%S", time.gmtime(stop_time - start_time)))

start_time = time.time()
params_et = {'n_estimators': 300, 'min_samples_leaf': 34, 'max_depth': 9, 'criterion': 'gini', 'class_weight': 'balanced', 'n_jobs':-1, 'random_state':1234}
et_clf = ExtraTreesClassifier(**params_et)
et_clf.fit(X_train_processed, y_train_processed)
print("Completed ExtraTreesClassifier")
stop_time = time.time()
print("Elapsed Time:", time.strftime("%H:%M:%S", time.gmtime(stop_time - start_time)))

start_time = time.time()
params_mlp =  {'solver': 'adam', 'max_iter': 1000, 'learning_rate': 'constant', 'hidden_layer_sizes': (50, 50), 'alpha': 0.0001, 'activation': 'relu', 'random_state':1234}
mlp_clf = MLPClassifier(**params_mlp)
mlp_clf.fit(X_train_processed, y_train_processed)
print("Completed MLP Classifier")
stop_time = time.time()
print("Elapsed Time:", time.strftime("%H:%M:%S", time.gmtime(stop_time - start_time)))


base_learners = [('random_forest', rf_clf),('extra_trees', et_clf),('mlp_classifier', mlp_clf)]

In [ ]:
start_time = time.time()

meta = LogisticRegression(max_iter=10000, class_weight='balanced')
stacked_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta,
    cv=5,
    passthrough=False,
    n_jobs=-1)

### LOAD THE STACKER
stacked_clf.fit(X_train_processed, y_train_processed)
print("Completed Stacked Classifier")
stop_time = time.time()
print("Elapsed Time:", time.strftime("%H:%M:%S", time.gmtime(stop_time - start_time)))

for name, X, y_true in [
    ("Validation", X_val_processed, y_val_processed),
    ("Training", X_train_processed, y_train_processed)]:
    y_pred = stacked_clf.predict(X)
    f1 = f1_score(y_true, y_pred, average='weighted')
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    conf_matrix = confusion_matrix(y_true, y_pred)
    print(f"\n=== {name} Set Results (Stacked Classifier) ===")
    print("Confusion Matrix:")
    print(conf_matrix)
    print(f"F1-score: {f1:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print("Classification Report:")
    print(classification_report(y_true, y_pred))

Here, we are implementing a model stacking solution setting up the models with the best parameters from previous model tuning, trying to use more complex methodology in a complex problem.

We divided this process in multiple steps:


1.   Training Individual (Base) Classifiers: Random Forest, Extra Trees and MLP Classifier
2.   Create a Stacked Classifier (final predictions are made by a logistic regression that learns from the outputs of the three base models.
3.   Making Predictions & Evaluate the model
4.   Evaluate the Metrics





# Model Comparison and Evaluation

When we look at the metrics of all our models, we can see very surprising results that we would not have imagined at the beginning of the project. The simplest models perform the best on the most relevant metric in this project - the F1 score. Even our simple OneRule Algorithm reaches a higher F1 score than a MLP Neural Network with optimized hyperparameters.

This strange pattern occurs due to problems in the dataset as we understood during the modeling process. Because of that, the best performance is reached when every instance in the validation set is predicted as class 1 "Survival".

While under normal circumstances we would choose the best 2-3 models according to F1 score at this stage, train them on the full dataset and then make predictions on the test set, these circumstances call for different solutions, as we have seen in our first Kaggle submission with a Model that classifies everything as class 1 that the model performance is weaker than the performance of our peers.

Therefore we will select not only the best models according to the F1 score to train on the full dataset and make predictions, but also include models with weaker performance which do not only classify every instance as class 1.

Eventually we will submit the predictions of all of these selected models on Kaggle and choose the one that scores the best on the test set as our final predictor.

**The models of our choice to try for the Kaggle submission are:**


1.  **Naive Bayes:**
As it's one of the models that has the highest F1 score (0.7505; predicting everything as class 1.
2.   **MLP Neural Network:** As among the models that classify a significant amount of instances as class 0 it shows one of the best F1 score performances (0.6753) and generally Neural Networks are well suited to solve complex problems.
3. **Stacking Model:** Even if it doesn’t show a very convincing performance on the validation set (F1 score: 0.57), we want to try this model on Kaggle as it predicts class 0 more frequently than other models and its complexity might help to solve the problem we are facing better than other models.
4. **Hyperparameter-tuned Random Forest:**
As among the hyperparameter-tuned ensemble classifiers is performs the best (given that class 0 is predicted a significant amount of times; F1 score: 0.6177).



### Process Test Dataset

In [ ]:
# In order to train our selected models on the whole dataset, we need to preprocess the data accordingly as we did for our initial train set.
##-----------------------------> PREPROCESS DATA 1
def preprocess_data(data):
    # Copy the data
    preprocessing_df = data.copy()

    # Remove ID Column
    #preprocessing_df = preprocessing_df.loc[:, preprocessing_df.columns != 'ID']

    # Duplicates removal
    preprocessing_df = preprocessing_df.drop_duplicates(keep='first')

    # Dataset split (Not applicable here)
    data_var = preprocessing_df.loc[:, preprocessing_df.columns != 'Survival Prediction']
    X_train = preprocessing_df
    # Outliers removal
    cols_to_remove_outliers = ['Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K', 'Tumor Size (mm)']
    for col in cols_to_remove_outliers:
        Q1 = X_train[col].quantile(0.25)
        Q3 = X_train[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        X_train = X_train[(X_train[col] >= lower_bound) & (X_train[col] <= upper_bound)]

    # Data inconsistencies
    X_train['Gender'] = X_train['Gender'].replace('P', 'M')
    X_train['Healthcare Access'] = X_train['Healthcare Access'].replace('?', np.nan)
    X_train.rename(columns={'Non Smoker': 'Smoker'}, inplace=True)
    X_train['Smoker'] = X_train['Smoker'].replace({'Yes': 'temp', 'No': 'Yes'})
    X_train['Smoker'] = X_train['Smoker'].replace({'temp': 'No'})
    X_train['Transfusion History'] = X_train['Transfusion History'].replace('-', np.nan)
    X_train['Urban or Rural'] = X_train['Urban or Rural'].str.lower()
    X_train['Urban or Rural'] = X_train['Urban or Rural'].replace({'urban': 'Urban', 'rural': 'Rural'})

    # Missing values
    X_train.drop(columns=['Marital Status', 'Transfusion History', 'Diabetes History', 'Smoking History'], inplace=True)
    cols_to_impute_numeric = ['Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K', 'Tumor Size (mm)']
    for col in cols_to_impute_numeric:
        X_train[col] = X_train[col].fillna(X_train[col].median())
    impute_with_mode_var = ['Country', 'Diabetes', 'Diet Risk', 'Early Detection', 'Family History', 'Gender', 'Genetic Mutation', 'Healthcare Access', 'Inflammatory Bowel Disease', 'Insurance Status', 'Urban or Rural']
    for col in impute_with_mode_var:
        X_train[col] = X_train[col].fillna(X_train[col].mode()[0])
    impute_with_random_var = ['Alcohol Consumption', 'Cancer Stage', 'Date of Birth', 'Obesity BMI', 'Physical Activity', 'Screening History', 'Treatment Type']
    for col in impute_with_random_var:
        non_missing = X_train[col].dropna().values
        X_train[col] = X_train[col].apply(lambda x: np.random.choice(non_missing) if pd.isna(x) else x)

    # Encoding
    yes_no_columns = ['Alcohol Consumption', 'Diabetes', 'Early Detection', 'Family History', 'Genetic Mutation', 'Heart Disease History', 'Hypertension', 'Inflammatory Bowel Disease', 'Smoker']
    for col in yes_no_columns:
        X_train[col] = X_train[col].replace({'No': 0, 'Yes': 1})
    X_train = pd.get_dummies(X_train, columns=['Gender', 'Insurance Status', 'Urban or Rural', 'Treatment Type'], drop_first=True)
    mappings = {
        'Diet Risk': {'Low': 0, 'Moderate': 1, 'High': 2},
        'Cancer Stage': {'Localized': 0, 'Regional': 1, 'Metastatic': 2},
        'Healthcare Access': {'Low': 0, 'Moderate': 1, 'High': 2},
        'Insurance Costs': {'No insurance': 0, 'Basic': 1, 'Extended': 2},
        'Obesity BMI': {'Normal': 0, 'Overweight': 1, 'Obese': 2},
        'Physical Activity': {'Low': 0, 'Moderate': 1, 'High': 2},
        'Screening History': {'Never': 0, 'Irregular': 1, 'Regular': 2}
    }
    for col, mapping in mappings.items():
        X_train[col] = X_train[col].replace(mapping)

    X_train['Country enc'] = X_train['Country'].map(country_target_mean_X_train)
    X_train.drop(columns=['Country'], inplace=True)

    # Feature Engineering
    X_train['Date of Birth'] = pd.to_datetime(X_train['Date of Birth'], format='%d-%m-%Y', errors='coerce')
    today = pd.to_datetime('today')
    X_train['Age'] = X_train['Date of Birth'].apply(lambda dob: today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day)) if pd.notnull(dob) else None)
    X_train['Age'] = X_train['Age'].astype('Int64')
    X_train.drop(columns=['Date of Birth'], inplace=True)
    X_train['Age_groups'] = pd.cut(X_train['Age'], bins=[0, 30, 60, 100], labels=[0, 1, 2])
    X_train['Tumor Size Categories'] = pd.cut(X_train['Tumor Size (mm)'], bins=[0, 35, 45, 100], labels=[0, 1, 2])
    X_train['Smoker and Alcohol Consumption'] = ((X_train['Smoker'] == 1) & (X_train['Alcohol Consumption'] == 1))
    comorbidity_cols = ['Diabetes', 'Heart Disease History', 'Hypertension', 'Inflammatory Bowel Disease']
    X_train['Comorbidity Count'] = X_train[comorbidity_cols].sum(axis=1)
    X_train['Mortality_to_Incidence_Ratio_per_100k'] = round(X_train['Mortality Rate per 100K'] / X_train['Incidence Rate per 100K'], 2)
    X_train['Family History and Genetic Mutation Interaction'] = ((X_train['Family History'] == 1) & (X_train['Genetic Mutation'] == 1))

    # Data Normalization
    numerical_columns = ['Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K', 'Tumor Size (mm)', 'Mortality_to_Incidence_Ratio_per_100k', 'Age']
    scaler = StandardScaler()
    scaler.fit(X_train[numerical_columns])
    X_train[numerical_columns] = scaler.transform(X_train[numerical_columns])

    # Final dataset preparation
    categorical_columns_m = ['Cancer Stage', 'Country enc', 'Diet Risk', 'Healthcare Access', 'Insurance Costs', 'Obesity BMI', 'Physical Activity', 'Screening History', 'Alcohol Consumption', 'Diabetes', 'Early Detection', 'Family History', 'Genetic Mutation', 'Heart Disease History', 'Hypertension', 'Inflammatory Bowel Disease', 'Smoker', 'Gender_M', 'Insurance Status_Uninsured', 'Urban or Rural_Urban', 'Treatment Type_Combination', 'Treatment Type_Radiotherapy', 'Treatment Type_Surgery']
    numerical_columns_m = ['Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K', 'Tumor Size (mm)']
    X_train_all = X_train[['ID'] + categorical_columns_m + numerical_columns_m]

    return X_train_all


new_data_test = pd.read_csv('/content/drive/MyDrive/DM2 Project/patient_test_data.csv')
original_dataset = pd.read_csv('/content/drive/MyDrive/DM2 Project/patient_train_data.csv')


target_var = original_dataset['Survival Prediction']
original_dataset.drop(columns=['Survival Prediction'], inplace=True)
original_dataset = preprocess_data(original_dataset)
original_dataset = original_dataset.drop(columns=["ID"])
original_dataset = original_dataset.drop_duplicates(keep='first')
target_var = target_var.loc[original_dataset.index]


##-----------------------------> PREPROCESS DATA 2
def preprocess_data2(data):
    # Copy the data
    X_train = data.copy()

    # Data inconsistencies
    X_train['Gender'] = X_train['Gender'].replace('P', 'M')
    X_train['Healthcare Access'] = X_train['Healthcare Access'].replace('?', np.nan)
    X_train.rename(columns={'Non Smoker': 'Smoker'}, inplace=True)
    X_train['Smoker'] = X_train['Smoker'].replace({'Yes': 'temp', 'No': 'Yes'})
    X_train['Smoker'] = X_train['Smoker'].replace({'temp': 'No'})
    X_train['Urban or Rural'] = X_train['Urban or Rural'].str.lower()
    X_train['Urban or Rural'] = X_train['Urban or Rural'].replace({'urban': 'Urban', 'rural': 'Rural'})

    # Missing values
    X_train.drop(columns=['Marital Status', 'Transfusion History', 'Diabetes History', 'Smoking History'], inplace=True)
    cols_to_impute_numeric = ['Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K', 'Tumor Size (mm)']
    for col in cols_to_impute_numeric:
        X_train[col] = X_train[col].fillna(X_train[col].median())
    impute_with_mode_var = ['Country', 'Diabetes', 'Diet Risk', 'Early Detection', 'Family History', 'Gender', 'Genetic Mutation', 'Healthcare Access', 'Inflammatory Bowel Disease', 'Insurance Status', 'Urban or Rural']
    for col in impute_with_mode_var:
        X_train[col] = X_train[col].fillna(X_train[col].mode()[0])
    impute_with_random_var = ['Alcohol Consumption', 'Cancer Stage', 'Date of Birth', 'Obesity BMI', 'Physical Activity', 'Screening History', 'Treatment Type']
    for col in impute_with_random_var:
        non_missing = X_train[col].dropna().values
        X_train[col] = X_train[col].apply(lambda x: np.random.choice(non_missing) if pd.isna(x) else x)

    # Encoding
    yes_no_columns = ['Alcohol Consumption', 'Diabetes', 'Early Detection', 'Family History', 'Genetic Mutation', 'Heart Disease History', 'Hypertension', 'Inflammatory Bowel Disease', 'Smoker']
    for col in yes_no_columns:
        X_train[col] = X_train[col].replace({'No': 0, 'Yes': 1})
    X_train = pd.get_dummies(X_train, columns=['Gender', 'Insurance Status', 'Urban or Rural', 'Treatment Type'], drop_first=True)
    mappings = {
        'Diet Risk': {'Low': 0, 'Moderate': 1, 'High': 2},
        'Cancer Stage': {'Localized': 0, 'Regional': 1, 'Metastatic': 2},
        'Healthcare Access': {'Low': 0, 'Moderate': 1, 'High': 2},
        'Insurance Costs': {'No insurance': 0, 'Basic': 1, 'Extended': 2},
        'Obesity BMI': {'Normal': 0, 'Overweight': 1, 'Obese': 2},
        'Physical Activity': {'Low': 0, 'Moderate': 1, 'High': 2},
        'Screening History': {'Never': 0, 'Irregular': 1, 'Regular': 2}
    }
    for col, mapping in mappings.items():
        X_train[col] = X_train[col].replace(mapping)

    X_train['Country enc'] = X_train['Country'].map(country_target_mean_X_train)
    X_train.drop(columns=['Country'], inplace=True)

    # Feature Engineering
    X_train['Date of Birth'] = pd.to_datetime(X_train['Date of Birth'], format='%d-%m-%Y', errors='coerce')
    today = pd.to_datetime('today')
    X_train['Age'] = X_train['Date of Birth'].apply(lambda dob: today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day)) if pd.notnull(dob) else None)
    X_train['Age'] = X_train['Age'].astype('Int64')
    X_train.drop(columns=['Date of Birth'], inplace=True)
    X_train['Age_groups'] = pd.cut(X_train['Age'], bins=[0, 30, 60, 100], labels=[0, 1, 2])
    X_train['Tumor Size Categories'] = pd.cut(X_train['Tumor Size (mm)'], bins=[0, 35, 45, 100], labels=[0, 1, 2])
    X_train['Smoker and Alcohol Consumption'] = ((X_train['Smoker'] == 1) & (X_train['Alcohol Consumption'] == 1))
    comorbidity_cols = ['Diabetes', 'Heart Disease History', 'Hypertension', 'Inflammatory Bowel Disease']
    X_train['Comorbidity Count'] = X_train[comorbidity_cols].sum(axis=1)
    X_train['Mortality_to_Incidence_Ratio_per_100k'] = round(X_train['Mortality Rate per 100K'] / X_train['Incidence Rate per 100K'], 2)
    X_train['Family History and Genetic Mutation Interaction'] = ((X_train['Family History'] == 1) & (X_train['Genetic Mutation'] == 1))

    # Data Normalization
    numerical_columns = ['Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K', 'Tumor Size (mm)', 'Mortality_to_Incidence_Ratio_per_100k', 'Age']
    scaler = StandardScaler()
    scaler.fit(X_train[numerical_columns])
    X_train[numerical_columns] = scaler.transform(X_train[numerical_columns])

    # Final dataset preparation
    categorical_columns_m = ['Cancer Stage', 'Country enc', 'Diet Risk', 'Healthcare Access', 'Insurance Costs', 'Obesity BMI', 'Physical Activity', 'Screening History', 'Alcohol Consumption', 'Diabetes', 'Early Detection', 'Family History', 'Genetic Mutation', 'Heart Disease History', 'Hypertension', 'Inflammatory Bowel Disease', 'Smoker', 'Gender_M', 'Insurance Status_Uninsured', 'Urban or Rural_Urban', 'Treatment Type_Combination', 'Treatment Type_Radiotherapy', 'Treatment Type_Surgery']
    numerical_columns_m = ['Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K', 'Tumor Size (mm)']
    X_train_all = X_train[['ID'] + categorical_columns_m + numerical_columns_m]

    return X_train_all

new_data_test = preprocess_data2(new_data_test)

### Naive Bayes Submission Model

In [ ]:
# Training our Naive Bayes Model on the entire dataset and receiving its prediction on the test dataset as a submission file.
nb = GaussianNB()
nb.fit(original_dataset, target_var)

y_pred = nb.predict(new_data_test.drop(columns=["ID"]))
y_pred_series = pd.Series(y_pred)

y_pred_series.replace({1: 'Yes', 0: 'No'}, inplace=True)
submission_nb = pd.DataFrame({
    "ID": new_data_test["ID"].astype(int),
    "Survival Prediction": y_pred_series})


submission_nb.to_csv('submission_nb.csv', index=False)
files.download('submission_nb.csv')

### MLP Submission Model


In [ ]:
# Training our MLP Neural Network Model on the entire dataset and receiving its prediction on the test dataset as a submission file.
mlp_clf = MLPClassifier(solver='adam',learning_rate='constant',hidden_layer_sizes=(100, 50),alpha=0.0001,activation='relu',max_iter=1000,random_state=1234)

mlp_clf.fit(original_dataset, target_var)

y_pred = mlp_clf.predict(new_data_test.drop(columns=["ID"]))
y_pred_series = pd.Series(y_pred)

y_pred_series.replace({1: 'Yes', 0: 'No'}, inplace=True)
submission_mlp = pd.DataFrame({
    "ID": new_data_test["ID"].astype(int),
    "Survival Prediction": y_pred_series})


submission_mlp.to_csv('submission_mlp.csv', index=False)
files.download('submission_mlp.csv')

### Stacking Submission Model

In [ ]:
# This chunk takes 28 minutes

# Training our Stacking Model on the entire dataset and receiving its prediction on the test dataset as a submission file.
stacked = StackingClassifier(estimators=base_learners)
stacked_clf = StackingClassifier(estimators=base_learners, final_estimator=RandomForestClassifier(), cv=10)

stacked_clf.fit(original_dataset, target_var)

y_pred = stacked_clf.predict(new_data_test.drop(columns=["ID"]))
y_pred_series = pd.Series(y_pred)

y_pred_series.replace({1: 'Yes', 0: 'No'}, inplace=True)
submission_stacked_clf = pd.DataFrame({
    "ID": new_data_test["ID"].astype(int),
    "Survival Prediction": y_pred_series})

submission_stacked_clf.to_csv('submission_stacked_clf.csv', index=False)
files.download('submission_stacked_clf.csv')

### Random Forest Final Submission

In [ ]:
# Training our hyperparameter-tuned Random Forest Model on the entire dataset and receiving its prediction on the test dataset as a submission file.

rf_clf = RandomForestClassifier(n_estimators= 800, min_samples_leaf= 32, max_features= 0.3, max_depth= 8, criterion= 'gini', class_weight= 'balanced', bootstrap= True)
rf_clf.fit(original_dataset, target_var)


y_pred = rf_clf.predict(new_data_test.drop(columns=["ID"]))
y_pred_series = pd.Series(y_pred)

y_pred_series.replace({1: 'Yes', 0: 'No'}, inplace=True)
submission_rf = pd.DataFrame({
    "ID": new_data_test["ID"].astype(int),
    "Survival Prediction": y_pred_series})


submission_rf.to_csv('submission_rf.csv', index=False)
files.download('submission_rf.csv')

###Kaggle Submission Results

After training these 4 models on the entire dataset, obtaining their predictions on the test set and uploading these predictions on Kaggle, we received the following scores for each model:


1.   Naive Bayes: 0.45360
2.   MLP Neural Network: 0.52743
3. Stacking Model: 0.52899
4. Hyperparameter-tuned Random Forest: 0.52177



###Choosing the final model

Due to the scores obtained from Kaggle, we chose the Stacking Model to be our final model that we are going to deploy to predict survival chances for colorectal cancer patients.

# Extra - Visualization Distributions with Target

In [ ]:
data_test = data.copy()

filtered_df_yes = data_test[data_test['Survival Prediction'] == 'Yes']
filtered_df_no = data_test[data_test['Survival Prediction'] == 'No']

In [ ]:
# Distribution plot for Tumor Size (mm) for both predictions
plt.figure(figsize=(10, 6))
sns.histplot(filtered_df_yes['Tumor Size (mm)'], color='green', label='Survival Prediction: Yes', kde=True, stat="density", alpha=0.6)
sns.histplot(filtered_df_no['Tumor Size (mm)'], color='red', label='Survival Prediction: No', kde=True, stat="density", alpha=0.6)

plt.xlabel('Tumor Size (mm)')
plt.ylabel('Density')
plt.title('Distribution of Tumor Size (mm) by Survival Prediction')
plt.legend()
plt.show()

sns.stripplot(
    data=data_test,
    x='Survival Prediction',
    y='Incidence Rate per 100K',
    palette={'Yes': 'green', 'No': 'red'},
    alpha=0.7,
    jitter=True
)
plt.title('Tumor Size (mm) by Survival Prediction')
plt.show()